# Ejari Automation Notebook

## Operator Notes

Run this notebook from the local network that can reach the DLD/DDA/DEWA APIs. Runtime outputs are intentionally kept brief; detailed API request curl files and response JSON files are written under `runs/` for each execution.

Security note: `.env`, `progress*.json`, failure reports, and run artifacts are ignored by git. The notebook redacts tokens in progress-style step logs, but curl repro files intentionally keep full headers so they can be shared with the API team when needed.


In [ ]:
# @title Shared helpers, operator prompts, and run artifact logging
RUN_OUTPUT_DIR = None
from notebook_config import (
    DDA_OWNER_EMIRATES_IDS,
    OWNER_EMIRATES_IDS,
    PERSONAL_OWNER_EMIRATES_ID,
)
from notebook_operator_utils import (
    ask_yes_no as ui_ask_yes_no,
    choose_option as ui_choose_option,
)
PROCESS_ALL_EMIRATES_IDS_BY_SECTION = {}


def safe_filename(value):
    import re
    safe_value = re.sub(r'[^a-zA-Z0-9_-]', '_', str(value or 'unknown'))
    return safe_value[:120]


def start_run_output_dir(prefix="ejari_creation"):
    import os
    from datetime import datetime

    global RUN_OUTPUT_DIR
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    RUN_OUTPUT_DIR = os.path.join("runs", f"{safe_filename(prefix)}_{timestamp}")
    os.makedirs(os.path.join(RUN_OUTPUT_DIR, "successes"), exist_ok=True)
    os.makedirs(os.path.join(RUN_OUTPUT_DIR, "failures"), exist_ok=True)
    os.makedirs(os.path.join(RUN_OUTPUT_DIR, "create_curls"), exist_ok=True)
    return RUN_OUTPUT_DIR


def get_run_output_dir():
    global RUN_OUTPUT_DIR
    if RUN_OUTPUT_DIR is None:
        RUN_OUTPUT_DIR = start_run_output_dir()
    return RUN_OUTPUT_DIR


def display_operator_prompt(prompt, default=None, choices=None):
    """Show a visible Jupyter prompt banner before falling back to stdin input."""
    choices_text = f"Choices: {', '.join(str(choice) for choice in choices)}" if choices else ""
    default_text = f"Default: {default}" if default is not None else ""
    try:
        import html
        from IPython.display import HTML, display
        detail_lines = [line for line in (choices_text, default_text) if line]
        detail_html = "<br>".join(html.escape(line) for line in detail_lines)
        display(HTML(f"""
        <div style="border:1px solid #b7791f;background:#fff8dc;color:#2d3748;padding:12px 14px;border-radius:6px;margin:8px 0;font-family:system-ui,Segoe UI,Arial,sans-serif;">
          <div style="font-weight:700;margin-bottom:4px;">Notebook is waiting for your input</div>
          <div>{html.escape(prompt)}</div>
          <div style="font-size:12px;color:#4a5568;margin-top:6px;">{detail_html}</div>
        </div>
        """))
    except Exception:
        print(f"\nNotebook is waiting for input: {prompt}")
        if choices_text:
            print(choices_text)
        if default_text:
            print(default_text)


def operator_input(prompt, default=None, choices=None, required=False, secret=False):
    import getpass

    while True:
        display_operator_prompt(prompt, default=default, choices=choices)
        suffix = f" [{default}]" if default is not None else ""
        raw_value = getpass.getpass(f"{prompt}{suffix}: ") if secret else input(f"{prompt}{suffix}: ")
        value = raw_value.strip()
        if value == "" and default is not None:
            value = str(default)
        if value or not required:
            return value
        print("A value is required.")


def ask_yes_no(prompt, default=False, yes_aliases=None, no_aliases=None):
    try:
        return ui_ask_yes_no(prompt, default=default, yes_aliases=yes_aliases, no_aliases=no_aliases)
    except Exception as exc:
        print(f"Popup selector unavailable; using typed input. {exc}")

    yes_values = {"y", "yes", "true", "1"} | set(yes_aliases or [])
    no_values = {"n", "no", "false", "0"} | set(no_aliases or [])
    default_text = "yes" if default else "no"
    while True:
        value = operator_input(f"{prompt} (yes/no)", default=default_text, choices=("yes", "no")).strip().lower()
        if value in yes_values:
            return True
        if value in no_values:
            return False
        print("Please enter yes or no.")


def ask_choice(prompt, choices, default=None, aliases=None):
    aliases = {str(key).strip().lower(): str(value) for key, value in (aliases or {}).items()}
    canonical = {str(choice).strip().lower(): str(choice) for choice in choices}
    if default is not None and str(default).strip().lower() not in canonical:
        raise ValueError(f"Default {default!r} is not in choices {choices!r}")

    try:
        selected = ui_choose_option(
            prompt,
            [(str(choice), str(choice)) for choice in choices],
            default=default,
            aliases=aliases,
            title="Select option",
        )
        selected = aliases.get(str(selected).strip().lower(), str(selected).strip().lower())
        if selected in canonical:
            return canonical[selected]
    except Exception as exc:
        print(f"Popup selector unavailable; using typed input. {exc}")

    while True:
        value = operator_input(prompt, default=default, choices=choices).strip().lower()
        value = aliases.get(value, value)
        if value in canonical:
            return canonical[value]
        print(f"Invalid choice. Choose one of: {', '.join(str(choice) for choice in choices)}")


def ask_int(prompt, default=None, min_value=None):
    while True:
        value = operator_input(prompt, default=default).strip()
        if value == "" and default is None:
            return None
        try:
            int_value = int(value)
        except ValueError:
            print("Please enter a whole number.")
            continue
        if min_value is not None and int_value < min_value:
            print(f"Please enter a value greater than or equal to {min_value}.")
            continue
        return int_value


def ask_process_all_emirates_ids(section_name, default=False):
    process_all = ask_yes_no(f"Process all Emirates IDs for {section_name}?", default=default, yes_aliases={"all"})
    PROCESS_ALL_EMIRATES_IDS_BY_SECTION[section_name] = process_all
    if process_all:
        print(f"Selected all configured Emirates IDs for {section_name}")
    return process_all


def confirm_process_emirates_id(section_name, emirates_id):
    if PROCESS_ALL_EMIRATES_IDS_BY_SECTION.get(section_name):
        return True

    if ask_yes_no(f"Process Emirates ID {emirates_id} for {section_name}?", default=True):
        return True
    print(f"Skipped {section_name} for Emirates ID {emirates_id}")
    return False



def print_processing_summary(section_name, emirates_id, total_count, unique_count, processed_count, success_count, failure_count):
    print(f"\n{section_name} summary for Emirates ID {emirates_id}:")
    print(f"total count: {total_count}")
    print(f"unique count: {unique_count}")
    print(f"processed count: {processed_count}")
    print(f"success count: {success_count}")
    print(f"failure count: {failure_count}")

def save_run_json(kind, emirates_id, title, data, property_id=None):
    import json
    import os
    from datetime import datetime

    run_dir = get_run_output_dir()
    subfolder = "successes" if kind == "success" else "failures"
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"{safe_filename(kind)}_{safe_filename(emirates_id)}_{safe_filename(property_id)}_{safe_filename(title)}_{timestamp}.json"
    path = os.path.join(run_dir, subfolder, filename)

    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    return path


SENSITIVE_LOG_FIELD_NAMES = {
    "authorization", "token", "cookie", "x-nv-header", "basic_auth", "client_secret",
    "id_token", "access_token", "refresh_token", "password", "secret"
}
SENSITIVE_LOG_FIELD_SUFFIXES = ("_token", "_secret", "_password")


def is_sensitive_log_field(field_name):
    key_text = str(field_name).lower()
    return key_text in SENSITIVE_LOG_FIELD_NAMES or key_text.endswith(SENSITIVE_LOG_FIELD_SUFFIXES)


def redact_headers(headers):
    return {
        key: "[REDACTED]" if is_sensitive_log_field(key) else value
        for key, value in (headers or {}).items()
    }


def redact_sensitive_values(value):
    if isinstance(value, dict):
        cleaned = {}
        for key, inner_value in value.items():
            if is_sensitive_log_field(key):
                cleaned[key] = "[REDACTED]"
            else:
                cleaned[key] = redact_sensitive_values(inner_value)
        return cleaned
    if isinstance(value, list):
        return [redact_sensitive_values(item) for item in value]
    return value


def shell_single_quote(value):
    return "'" + str(value).replace("'", "'\"'\"'") + "'"


def build_curl_command(method, url, headers, payload=None):
    import json

    line_continuation = " " + chr(92) + "\n"
    curl = f"curl --location --request {method.upper()} {shell_single_quote(url)}" + line_continuation
    for key, value in (headers or {}).items():
        curl += f"--header {shell_single_quote(f'{key}: {value}')}" + line_continuation
    if payload is not None:
        curl += f"--data-raw {shell_single_quote(json.dumps(payload, ensure_ascii=False))}"
    else:
        curl = curl[:-len(line_continuation)] if curl.endswith(line_continuation) else curl
    return curl


def save_api_curl_file(subfolder, filename_prefix, emirates_id, title, method, url, headers, payload=None, property_id=None, timestamp=None):
    import os
    from datetime import datetime

    timestamp = timestamp or datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"{safe_filename(filename_prefix)}_{safe_filename(emirates_id)}_{safe_filename(property_id)}_{safe_filename(title)}_{timestamp}.sh"
    path = os.path.join(get_run_output_dir(), subfolder, filename)
    os.makedirs(os.path.dirname(path), exist_ok=True)

    with open(path, "w", encoding="utf-8") as f:
        f.write(build_curl_command(method, url, headers, payload))

    return path


def save_api_response_json(kind, api_name, emirates_id, title, result, property_id=None, timestamp=None):
    import json
    import os
    from datetime import datetime

    timestamp = timestamp or datetime.now().strftime("%Y%m%d_%H%M%S")
    subfolder = "successes" if kind == "success" else "failures"
    filename_prefix = "success" if kind == "success" else "failed"
    filename = f"{filename_prefix}_{safe_filename(api_name)}_response_{safe_filename(emirates_id)}_{safe_filename(property_id)}_{safe_filename(title)}_{timestamp}.json"
    path = os.path.join(get_run_output_dir(), subfolder, filename)
    os.makedirs(os.path.dirname(path), exist_ok=True)

    result = result if isinstance(result, dict) else {"response": result}
    detail = {
        "api_name": api_name,
        "kind": kind,
        "status_code": result.get("status_code"),
        "url": result.get("url"),
        "response": result.get("response"),
        "timestamp": datetime.now().isoformat()
    }
    with open(path, "w", encoding="utf-8") as f:
        json.dump(detail, f, ensure_ascii=False, indent=2)

    return path


def save_api_detail_files(kind, api_name, emirates_id, title, method, url, headers, payload, result, property_id=None):
    from datetime import datetime

    if not url:
        return {}

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    subfolder = "successes" if kind == "success" else "failures"
    filename_prefix = f"{'success' if kind == 'success' else 'failed'}_{safe_filename(api_name)}"
    curl_path = save_api_curl_file(subfolder, filename_prefix, emirates_id, title, method, url, headers, payload, property_id, timestamp)
    response_path = save_api_response_json(kind, api_name, emirates_id, title, result, property_id, timestamp)
    return {
        "api_request_curl_path": curl_path,
        "api_response_path": response_path
    }


def save_api_result_detail_files(kind, api_name, emirates_id, title, method, result, property_id=None):
    result = result if isinstance(result, dict) else {}
    return save_api_detail_files(
        kind,
        api_name,
        emirates_id,
        title,
        method,
        result.get("url"),
        result.get("headers"),
        result.get("payload"),
        result,
        property_id
    )


def save_api5_detail_files(kind, emirates_id, title, url, headers, payload, result, property_id=None):
    detail_paths = save_api_detail_files(kind, "api5", emirates_id, title, "post", url, headers, payload, result, property_id)
    if detail_paths:
        detail_paths["api5_request_curl_path"] = detail_paths.get("api_request_curl_path")
        detail_paths["api5_response_path"] = detail_paths.get("api_response_path")
    return detail_paths


def save_failed_api_curl(api_name, emirates_id, title, method, url, headers, payload=None, property_id=None):
    return save_api_curl_file("failures", f"failed_{safe_filename(api_name)}", emirates_id, title, method, url, headers, payload, property_id)


def save_failed_api4(emirates_id, title, url, headers, property_id=None):
    return save_failed_api_curl("api4", emirates_id, title, "get", url, headers, property_id=property_id)


def save_failed_api5(emirates_id, title, url, headers, payload, property_id=None):
    return save_failed_api_curl("api5", emirates_id, title, "post", url, headers, payload, property_id)


def save_create_api5_curl(emirates_id, title, url, headers, payload, property_id=None):
    return save_api_curl_file("create_curls", "create_api5", emirates_id, title, "post", url, headers, payload, property_id)


In [ ]:
# @title API definitions, config, storage, and shared workflow helpers
##########################################################################################
####################### API DEFINITIONS ##################################################
##########################################################################################


from contextlib import nullcontext
import requests
import json
import base64
import os
from datetime import datetime

try:
    import google.colab.userdata as userdata
except ImportError:
    userdata = None

def load_local_env(path='.env'):
    if not os.path.exists(path):
        return
    with open(path, 'r', encoding='utf-8') as f:
        for raw_line in f:
            line = raw_line.strip()
            if not line or line.startswith('#') or '=' not in line:
                continue
            key, value = line.split('=', 1)
            key = key.strip()
            value = value.strip().strip('\"').strip("'")
            os.environ.setdefault(key, value)

load_local_env()

def get_secret(name, default=None):
    if userdata is not None:
        value = userdata.get(name)
        if value is not None:
            return value
    return os.environ.get(name, default)

# ================= CONFIG =================
def require_secret(name):
    value = get_secret(name)
    if value is None or str(value).strip() == "":
        raise RuntimeError(f"Missing required secret/env var: {name}")
    return str(value).strip()


def optional_secret(name, default=None):
    value = get_secret(name, default)
    if value is None:
        return None
    text = str(value).strip()
    return text or None


def parse_positive_int(value, default):
    try:
        parsed = int(value)
    except (TypeError, ValueError):
        return default
    return parsed if parsed > 0 else default


BASIC_AUTH = require_secret('BASIC_AUTH')
CONSUMER_ID = require_secret('CONSUMER_ID')
DLD_BASE_URL = require_secret('DLD_BASE_URL').rstrip('/')
DLD_PROXY_URL = require_secret('DLD_PROXY_URL').rstrip('/')
IDS_BASE_URL = require_secret('IDS_BASE_URL').rstrip('/')

DEWA_BASE_URL = (optional_secret('DEWA_BASE_URL') or '').rstrip('/')
DEWA_CLIENT_ID = optional_secret('DEWA_CLIENT_ID')
DEWA_CLIENT_SECRET = optional_secret('DEWA_CLIENT_SECRET')
DEWA_VENDOR_ID = optional_secret('DEWA_VENDOR_ID')
REQUEST_TIMEOUT_SECONDS = parse_positive_int(optional_secret('REQUEST_TIMEOUT_SECONDS'), 90)

STORAGE_FILE = "progress.json"
CURRENT_BEARER_TOKEN = None

# ================= STORAGE =================
def normalize_progress_property(prop_data, property_key=None):
    if "ejari_id" in prop_data and "ejari_property_id" not in prop_data:
        prop_data["ejari_property_id"] = prop_data.pop("ejari_id")
    else:
        prop_data.pop("ejari_id", None)

    if "ejariId" in prop_data and "ejari_property_id" not in prop_data:
        prop_data["ejari_property_id"] = prop_data.pop("ejariId")
    else:
        prop_data.pop("ejariId", None)

    if "row_value" in prop_data and "property_row_value" not in prop_data:
        prop_data["property_row_value"] = prop_data.pop("row_value")
    else:
        prop_data.pop("row_value", None)

    if "propertyId" in prop_data and "property_id" not in prop_data:
        prop_data["property_id"] = prop_data.pop("propertyId")
    else:
        prop_data.pop("propertyId", None)

    if prop_data.get("property_id") is None:
        candidate = prop_data.get("property_row_value") or property_key
        if candidate is not None and str(candidate).isdigit():
            prop_data["property_id"] = int(candidate)

    if prop_data.get("status") == "ERROR" and not prop_data.get("api_stage"):
        prop_data["api_stage"] = "API5_CREATE_CONTRACT" if prop_data.get("failed_api5_id") else "API4_VALIDATE_PROPERTY_OR_PRE_API5"

    return prop_data

def normalize_property_counts(emirates_data):
    counts = emirates_data.get("property_counts")
    if not isinstance(counts, dict):
        return

    vacant_villas = counts.get("vacant/2", 0)
    vacant_units = counts.get("vacant/3", 0)
    emirates_data["property_counts"] = {
        "vacant/2": vacant_villas,
        "vacant/3": vacant_units,
        "total_vacant": vacant_villas + vacant_units
    }

def normalize_progress_schema(progress):
    for emirates_data in progress.values():
        normalize_property_counts(emirates_data)
        success = emirates_data.setdefault("ejari_creation_success", {})
        failure = emirates_data.setdefault("ejari_creation_failure", {})
        curl_only = emirates_data.setdefault("ejari_creation_curl_only", {})

        legacy_properties = emirates_data.pop("properties", {})
        for property_key, prop_data in legacy_properties.items():
            normalize_progress_property(prop_data, property_key)
            status = prop_data.get("status")
            if status == "SUCCESS":
                success[str(property_key)] = prop_data
            elif status == "ERROR":
                failure[str(property_key)] = prop_data
            else:
                failure[str(property_key)] = prop_data

        for property_key, prop_data in success.items():
            normalize_progress_property(prop_data, property_key)
        for property_key, prop_data in failure.items():
            normalize_progress_property(prop_data, property_key)
        for property_key, prop_data in curl_only.items():
            normalize_progress_property(prop_data, property_key)

    return progress

def ensure_emirates_progress(progress, emirates_id):
    if emirates_id not in progress:
        progress[emirates_id] = {}
    progress[emirates_id].setdefault("ejari_creation_success", {})
    progress[emirates_id].setdefault("ejari_creation_failure", {})
    progress[emirates_id].setdefault("ejari_creation_curl_only", {})
    return progress[emirates_id]

def format_failure_api_error(value):
    if value is None or value == "":
        return ""

    parsed_value = value
    if isinstance(value, str):
        text = value.strip()
        if text.startswith("{") or text.startswith("["):
            try:
                parsed_value = json.loads(text)
            except Exception:
                return text
        else:
            return text

    if not isinstance(parsed_value, dict):
        return json.dumps(parsed_value, ensure_ascii=False)

    response_code = parsed_value.get("ResponseCode")
    validation_errors = parsed_value.get("ValidationErrorsList") or []
    formatted_errors = []
    for api_error in validation_errors:
        if not isinstance(api_error, dict):
            formatted_errors.append(str(api_error))
            continue

        message_set = api_error.get("ErrorMessageSet") or {}
        english_name = message_set.get("EnglishName") if isinstance(message_set, dict) else None
        message = english_name or api_error.get("ErrorMessage")
        error_number = api_error.get("ErrorNumber")
        error_parts = []
        if error_number is not None:
            error_parts.append(f"ErrorNumber={error_number}")
        if message:
            error_parts.append(f"Message={message}")
        if error_parts:
            formatted_errors.append("; ".join(error_parts))

    if formatted_errors:
        prefix = f"ResponseCode={response_code}; " if response_code else ""
        return prefix + " | ".join(formatted_errors)

    return json.dumps(parsed_value, ensure_ascii=False)


def split_failure_report_error(prop_data):
    if prop_data.get("automation_error") is not None or prop_data.get("api_error") is not None:
        return prop_data.get("automation_error") or "", format_failure_api_error(prop_data.get("api_error"))

    error = str(prop_data.get("error") or "")
    error = error.split(" | fail_id=", 1)[0]
    api_stage = prop_data.get("api_stage") or ""
    split_prefixes = (
        "API3 get properties failed HTTP",
        "API3 get properties returned unexpected payload",
        "API4 validate failed HTTP",
        "API4 validate returned no EjariPropertyIDs",
        "API5 HTTP",
        "API5 returned non-JSON response"
    )

    if error.startswith(split_prefixes) and ": " in error:
        automation_error, api_error = error.split(": ", 1)
        return automation_error, format_failure_api_error(api_error)

    if api_stage == "API5_CREATE_CONTRACT" and prop_data.get("failed_api5_id"):
        return "", format_failure_api_error(error)

    return error, ""


DLD_FAILURE_REPORT_FIELDS = [
    "emirates_id",
    "property_id",
    "ejari_property_id",
    "property_row_value",
    "title",
    "status",
    "api_stage",
    "failed_step",
    "step_status",
    "attempt_count",
    "last_attempt",
    "http_status_code",
    "token_source",
    "tenant_emirates_id",
    "tenant_id",
    "tenant_name",
    "contract_row_value",
    "contract_number",
    "automation_error",
    "api_error",
    "error",
    "failed_api4_id",
    "failed_api4_path",
    "failed_api5_id",
    "failed_api5_path",
    "api_request_curl_path",
    "api_response_path",
    "api5_request_curl_path",
    "api5_response_path",
    "failure_json_path",
    "timestamp"
]


def get_step_final_result(step):
    if not isinstance(step, dict):
        return None
    attempts = step.get("attempts") or []
    return step.get("result") or (attempts[-1] if attempts else None)


def step_failed(step):
    if not isinstance(step, dict):
        return False
    if step.get("status") == "ERROR":
        return True
    if step.get("status") in {"SUCCESS", "SKIPPED"}:
        return False
    result = get_step_final_result(step)
    return result is not None and not api_result_succeeded(result)


def split_step_report_error(step):
    if not isinstance(step, dict):
        return "Invalid step record", ""
    result = get_step_final_result(step) or {}
    automation_error = step.get("automation_error") or result.get("error") or ""
    api_error = step.get("api_error") or ""
    if not api_error and isinstance(result, dict):
        payload_error = describe_api_payload_errors(result.get("response"))
        formatted_payload = format_failure_api_error(result.get("response"))
        api_error = payload_error
        if formatted_payload and formatted_payload not in {"null", api_error}:
            api_error = (api_error + " | " if api_error else "") + formatted_payload
    status_code = result.get("status_code") if isinstance(result, dict) else None
    if status_code is not None and status_code >= 400:
        automation_error = (automation_error + "; " if automation_error else "") + f"HTTP {status_code}"
    return automation_error, api_error


def build_creation_step_report_rows(emirates_id, property_key, prop_data):
    rows = []
    for step in prop_data.get("step_results") or []:
        if not step_failed(step):
            continue
        result = get_step_final_result(step) or {}
        attempts = step.get("attempts") or []
        automation_error, api_error = split_step_report_error(step)
        rows.append({
            "emirates_id": emirates_id,
            "property_id": prop_data.get("property_id"),
            "ejari_property_id": prop_data.get("ejari_property_id"),
            "property_row_value": prop_data.get("property_row_value") or property_key,
            "title": prop_data.get("title"),
            "status": prop_data.get("status"),
            "api_stage": prop_data.get("api_stage"),
            "failed_step": step.get("api_stage"),
            "step_status": step.get("status"),
            "attempt_count": step.get("attempt_count") or len(attempts) or (1 if result else 0),
            "last_attempt": result.get("attempt") if isinstance(result, dict) else None,
            "http_status_code": result.get("status_code") if isinstance(result, dict) else None,
            "token_source": step.get("token_source"),
            "tenant_emirates_id": step.get("tenant_emirates_id") or prop_data.get("tenant_emirates_id"),
            "tenant_id": step.get("tenant_id") or prop_data.get("tenant_id"),
            "tenant_name": prop_data.get("tenant_name"),
            "contract_row_value": prop_data.get("contract_row_value"),
            "contract_number": prop_data.get("contract_number"),
            "automation_error": automation_error,
            "api_error": api_error,
            "error": prop_data.get("error"),
            "failed_api4_id": prop_data.get("failed_api4_id"),
            "failed_api4_path": prop_data.get("failed_api4_id"),
            "failed_api5_id": prop_data.get("failed_api5_id"),
            "failed_api5_path": prop_data.get("failed_api5_id"),
            "api_request_curl_path": step.get("api_request_curl_path") or (step.get("result") or {}).get("request_curl_path") or prop_data.get("api5_request_curl_path") or prop_data.get("failed_api5_id"),
            "api_response_path": step.get("api_response_path") or (step.get("result") or {}).get("response_json_path") or prop_data.get("api5_response_path"),
            "api5_request_curl_path": prop_data.get("api5_request_curl_path") or prop_data.get("failed_api5_id"),
            "api5_response_path": prop_data.get("api5_response_path"),
            "failure_json_path": prop_data.get("failure_json_path"),
            "timestamp": prop_data.get("timestamp")
        })
    return rows


def build_failure_report(progress):
    rows = []
    for emirates_id, emirates_data in progress.items():
        failure_records = emirates_data.get("ejari_creation_failure", {})
        for property_key, prop_data in failure_records.items():
            step_rows = build_creation_step_report_rows(emirates_id, property_key, prop_data)
            if step_rows:
                rows.extend(step_rows)
                continue
            failed_api4_id = prop_data.get("failed_api4_id")
            failed_api5_id = prop_data.get("failed_api5_id")
            automation_error, api_error = split_failure_report_error(prop_data)
            rows.append({
                "emirates_id": emirates_id,
                "property_id": prop_data.get("property_id"),
                "ejari_property_id": prop_data.get("ejari_property_id"),
                "property_row_value": prop_data.get("property_row_value") or property_key,
                "title": prop_data.get("title"),
                "status": prop_data.get("status"),
                "api_stage": prop_data.get("api_stage"),
                "failed_step": prop_data.get("api_stage"),
                "step_status": prop_data.get("status"),
                "automation_error": automation_error,
                "api_error": api_error,
                "error": prop_data.get("error"),
                "tenant_emirates_id": prop_data.get("tenant_emirates_id"),
                "tenant_id": prop_data.get("tenant_id"),
                "tenant_name": prop_data.get("tenant_name"),
                "failed_api4_id": failed_api4_id,
                "failed_api4_path": failed_api4_id,
                "failed_api5_id": failed_api5_id,
                "failed_api5_path": failed_api5_id,
                "api_request_curl_path": prop_data.get("api5_request_curl_path") or failed_api5_id,
                "api_response_path": prop_data.get("api5_response_path"),
                "api5_request_curl_path": prop_data.get("api5_request_curl_path") or failed_api5_id,
                "api5_response_path": prop_data.get("api5_response_path"),
                "failure_json_path": prop_data.get("failure_json_path"),
                "timestamp": prop_data.get("timestamp")
            })
    return rows


def save_failure_report(progress, output_dir="."):
    import csv
    rows = build_failure_report(progress)
    os.makedirs(output_dir, exist_ok=True)
    json_path = os.path.join(output_dir, "failure_report.json")
    csv_path = os.path.join(output_dir, "failure_report.csv")
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(rows, f, ensure_ascii=False, indent=2)
    try:
        with open(csv_path, "w", encoding="utf-8", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=DLD_FAILURE_REPORT_FIELDS, extrasaction="ignore")
            writer.writeheader()
            writer.writerows(rows)
    except PermissionError:
        print(f"Could not update {csv_path}. Close the CSV file in Excel before reading the latest failure report.")

def backfill_progress_property_ids(emirates_progress, properties):
    record_sets = [
        emirates_progress.get("ejari_creation_success", {}),
        emirates_progress.get("ejari_creation_failure", {}),
        emirates_progress.get("ejari_creation_curl_only", {})
    ]

    for prop in properties:
        property_id = prop.get("PropertyId")
        row_value = prop.get("RowValue") or property_id
        if property_id is None or row_value is None:
            continue

        property_key = str(row_value)
        for records in record_sets:
            prop_data = records.get(property_key)
            if prop_data:
                prop_data["property_id"] = property_id
                prop_data["property_row_value"] = row_value

def load_progress():
    if os.path.exists(STORAGE_FILE):
        with open(STORAGE_FILE, "r", encoding="utf-8") as f:
            return normalize_progress_schema(json.load(f))
    return {}

def save_progress(data):
    normalized_data = normalize_progress_schema(data)
    with open(STORAGE_FILE, "w", encoding="utf-8") as f:
        json.dump(normalized_data, f, indent=2)
    save_failure_report(normalized_data)

    run_output_dir = globals().get("RUN_OUTPUT_DIR")
    if run_output_dir:
        os.makedirs(run_output_dir, exist_ok=True)
        with open(os.path.join(run_output_dir, STORAGE_FILE), "w", encoding="utf-8") as f:
            json.dump(normalized_data, f, indent=2)
        save_failure_report(normalized_data, run_output_dir)

# ================= API1 =================
def get_bearer_token():
    url = IDS_BASE_URL

    headers = {
        "Authorization": f"Basic {BASIC_AUTH}",
        "Content-Type": "application/x-www-form-urlencoded"
    }

    response = requests.post(url, headers=headers, data={}, timeout=REQUEST_TIMEOUT_SECONDS)
    response.raise_for_status()

    token = response.json()["id_token"]

    global CURRENT_BEARER_TOKEN
    CURRENT_BEARER_TOKEN = token
    return token

def get_active_bearer_token(force_refresh=False):
    global CURRENT_BEARER_TOKEN
    if force_refresh or not CURRENT_BEARER_TOKEN:
        return get_bearer_token()
    return CURRENT_BEARER_TOKEN

def request_with_bearer(method, url, headers=None, retry_on_401=True, **kwargs):
    request_headers = dict(headers or {})
    request_headers["Authorization"] = f"Bearer {get_active_bearer_token()}"
    kwargs.setdefault("timeout", REQUEST_TIMEOUT_SECONDS)
    response = requests.request(method, url, headers=request_headers, **kwargs)

    if response.status_code == 401 and retry_on_401:
        print("iPaaS bearer token expired; refreshing and retrying request once.")
        request_headers["Authorization"] = f"Bearer {get_active_bearer_token(force_refresh=True)}"
        response = requests.request(method, url, headers=request_headers, **kwargs)

    return response

def decode_json_like(value):
    if isinstance(value, str):
        text = value.strip()
        if text.startswith("{") or text.startswith("["):
            try:
                return decode_json_like(json.loads(text))
            except Exception:
                return value
        return value
    if isinstance(value, dict):
        return {key: decode_json_like(inner_value) for key, inner_value in value.items()}
    if isinstance(value, list):
        return [decode_json_like(item) for item in value]
    return value

def response_payload(response):
    try:
        return decode_json_like(response.json())
    except Exception:
        return decode_json_like(response.text)


def final_request_auth_headers(response, fallback_headers):
    final_headers = dict(fallback_headers or {})
    request = getattr(response, "request", None)
    sent_headers = getattr(request, "headers", {}) or {}
    sensitive_header_names = {"authorization", "token"}

    for sent_key, sent_value in sent_headers.items():
        if str(sent_key).lower() not in sensitive_header_names:
            continue
        existing_key = next((key for key in final_headers if str(key).lower() == str(sent_key).lower()), sent_key)
        final_headers[existing_key] = sent_value
    return final_headers

def compact_payload(value, max_chars=2000):
    if isinstance(value, str):
        text = value
    else:
        text = json.dumps(value, ensure_ascii=False)
    return text if len(text) <= max_chars else text[:max_chars] + "..."

def get_api5_error_message(error):
    if not isinstance(error, dict):
        return str(error)

    message_set = error.get("ErrorMessageSet") or {}
    if isinstance(message_set, dict):
        message = message_set.get("EnglishName") or message_set.get("ArabicName")
    else:
        message = str(message_set)

    return message or compact_payload(error)

def api_payload_errors(payload):
    if not isinstance(payload, dict):
        return [{"ErrorNumber": None, "ErrorMessageSet": {"EnglishName": "API response payload is not a JSON object."}, "ValidationType": "ResponsePayload"}]
    errors = []
    response_code = payload.get("ResponseCode")
    if response_code is not None and str(response_code).strip().lower() != "success":
        errors.append({"ErrorNumber": None, "ErrorMessageSet": {"EnglishName": f"ResponseCode={response_code}"}, "ValidationType": "ResponseCode"})
    if payload.get("ErrorMessage"):
        errors.append({"ErrorNumber": None, "ErrorMessageSet": {"EnglishName": str(payload.get("ErrorMessage"))}, "ValidationType": "ErrorMessage"})
    for api_error in payload.get("ValidationErrorsList") or []:
        if not isinstance(api_error, dict):
            errors.append({"ErrorNumber": None, "ErrorMessageSet": {"EnglishName": str(api_error)}, "ValidationType": "ValidationErrorsList"})
            continue
        error_number = api_error.get("ErrorNumber")
        message_set = api_error.get("ErrorMessageSet") or {}
        message = api_error.get("ErrorMessage") or (message_set.get("EnglishName") if isinstance(message_set, dict) else None)
        if error_number not in (None, 0, "0"):
            errors.append(api_error)
        elif message and str(message).strip().lower() not in {"no errors found", "no errors found."}:
            errors.append(api_error)
    return errors


def describe_api_payload_errors(payload):
    return " | ".join(get_api5_error_message(api_error) for api_error in api_payload_errors(payload))


def api_result_succeeded(result):
    if not isinstance(result, dict):
        return False
    status_code = result.get("status_code")
    if status_code is None or status_code >= 400:
        return False
    return not api_payload_errors(result.get("response"))


def dld_token_result_succeeded(result):
    if not isinstance(result, dict):
        return False
    response = result.get("response")
    return result.get("status_code") is not None and result.get("status_code") < 400 and isinstance(response, dict) and bool(response.get("token"))


def result_error_summary(result):
    if not isinstance(result, dict):
        return "Missing API result."
    pieces = []
    status_code = result.get("status_code")
    if status_code is None:
        pieces.append("HTTP status missing")
    elif status_code >= 400:
        pieces.append(f"HTTP {status_code}")
    if result.get("error"):
        pieces.append(str(result.get("error")))
    api_errors = describe_api_payload_errors(result.get("response"))
    if api_errors:
        pieces.append(api_errors)
    elif result.get("response") not in (None, "") and not api_result_succeeded(result):
        pieces.append(format_failure_api_error(result.get("response")))
    return " | ".join(piece for piece in pieces if piece) or "No API error captured."


def strip_large_contract_file_fields(value):
    file_keys = {"contractfile", "contract_file", "filecontent", "file_content", "document", "pdf", "reportdata", "report_data"}
    if isinstance(value, dict):
        cleaned = {}
        for key, inner_value in value.items():
            if str(key).lower() in file_keys and isinstance(inner_value, str):
                cleaned[key] = f"[base64 content length={len(inner_value)}]"
            else:
                cleaned[key] = strip_large_contract_file_fields(inner_value)
        return cleaned
    if isinstance(value, list):
        return [strip_large_contract_file_fields(item) for item in value]
    return value


def log_safe_api_result(result):
    if not isinstance(result, dict):
        return result
    return redact_sensitive_values(strip_large_contract_file_fields(result))


def make_api_step(api_stage, result=None, status=None, attempts=None, automation_error=None, api_error=None, **extra):
    final_result = result
    if final_result is None and attempts:
        final_result = attempts[-1]
    if status is None:
        status = "SUCCESS" if api_result_succeeded(final_result) else "ERROR"
    step = {"api_stage": api_stage, "status": status, "timestamp": datetime.now().isoformat()}
    if result is not None:
        step["result"] = log_safe_api_result(result)
    if attempts is not None:
        step["attempts"] = [log_safe_api_result(attempt) for attempt in attempts]
        step["attempt_count"] = len(attempts)
        if attempts:
            step["result"] = log_safe_api_result(attempts[-1])
    if automation_error:
        step["automation_error"] = automation_error
    if api_error:
        step["api_error"] = api_error
    step.update(extra)
    return step


def api_attempt_reference(result):
    result = result if isinstance(result, dict) else {}
    keep_keys = (
        "status_code", "url", "endpoint_name", "attempt", "max_attempts", "error",
        "token_source", "tenant_emirates_id", "tenant_sequence", "owner_asset_list_type",
        "source_owner_asset_list_type", "property_type_id", "property_row_value"
    )
    return {key: result.get(key) for key in keep_keys if result.get(key) is not None}


def api_detail_reference_result(result, detail_paths):
    result = result if isinstance(result, dict) else {}
    detail_paths = detail_paths or {}
    reference = api_attempt_reference(result)
    reference["request_curl_path"] = detail_paths.get("api_request_curl_path") or detail_paths.get("api5_request_curl_path")
    reference["response_json_path"] = detail_paths.get("api_response_path") or detail_paths.get("api5_response_path")
    return reference


def attach_api_detail_paths_to_step(step, api_result, detail_paths):
    if not isinstance(step, dict) or not detail_paths:
        return
    step["result"] = api_detail_reference_result(api_result, detail_paths)
    if step.get("attempts"):
        step["attempts"] = [api_attempt_reference(attempt) for attempt in step.get("attempts") or []]
    step["api_request_curl_path"] = detail_paths.get("api_request_curl_path")
    step["api_response_path"] = detail_paths.get("api_response_path")
    api_error = describe_api_payload_errors(api_result.get("response") if isinstance(api_result, dict) else None)
    if api_error:
        step["api_error"] = api_error


def attach_api5_detail_paths_to_latest_step(step_results, api5_result, detail_paths):
    if not step_results:
        return
    step = step_results[-1]
    if not isinstance(step, dict) or step.get("api_stage") != "API5_CREATE_CONTRACT":
        return
    attach_api_detail_paths_to_step(step, api5_result, detail_paths)
    step["api5_request_curl_path"] = detail_paths.get("api5_request_curl_path")
    step["api5_response_path"] = detail_paths.get("api5_response_path")


def save_and_attach_api_step_detail(step, api_name, emirates_id, title, method, result, property_id=None):
    if not isinstance(step, dict) or not isinstance(result, dict) or not result.get("url"):
        return {}
    kind = "success" if step.get("status") == "SUCCESS" and api_result_succeeded(result) else "failure"
    detail_paths = save_api_result_detail_files(kind, api_name, emirates_id, title, method, result, property_id)
    attach_api_detail_paths_to_step(step, result, detail_paths)
    return detail_paths


def run_api_with_retries(api_stage, call_func, attempts=3, delay_seconds=2, success_check=api_result_succeeded):
    import time
    raw_attempts = []
    for attempt_number in range(1, attempts + 1):
        try:
            result = call_func()
        except Exception as exc:
            result = {"status_code": None, "response": None, "error": str(exc)}
        if isinstance(result, dict):
            result["attempt"] = attempt_number
            result["max_attempts"] = attempts
        raw_attempts.append(result)
        if success_check(result):
            return make_api_step(api_stage, attempts=raw_attempts, status="SUCCESS"), result
        if attempt_number < attempts:
            print(f"{api_stage} failed on attempt {attempt_number}/{attempts}; retrying in {delay_seconds} seconds.")
            time.sleep(delay_seconds)
    return make_api_step(api_stage, attempts=raw_attempts, status="ERROR"), (raw_attempts[-1] if raw_attempts else None)


def raise_api_step_failure(api_stage, result, message):
    error_text = result_error_summary(result)
    raise ApiRequestError(f"{message}. {error_text}", api_stage, result.get("url") if isinstance(result, dict) else None, result.get("headers") if isinstance(result, dict) else None, result=result)


class ApiRequestError(Exception):
    def __init__(self, message, api_stage=None, url=None, headers=None, result=None):
        super().__init__(message)
        self.api_stage = api_stage
        self.url = url
        self.headers = headers
        self.result = result

# ================= API2 =================
def build_dld_token_request(emirates_id, bearer_token):
    url = f"{DLD_BASE_URL}/generaltokenservice/1.0.0/auth"
    auth_obj = {
        "Method": "EmiratesId",
        "MethodIdentity": str(emirates_id),
        "MethodPasscode": "",
        "DeviceKey": "MyPC",
        "ApplicationKey": "DubaiNow",
        "Platform": "Mobile"
    }
    encoded = base64.b64encode(json.dumps(auth_obj).encode()).decode()
    headers = {
        "consumer-id": CONSUMER_ID,
        "x-nv-header": encoded,
        "Authorization": f"Bearer {bearer_token}",
        "Content-Type": "application/json"
    }
    return url, headers, auth_obj


def get_dld_token_result(emirates_id, bearer_token):
    url, headers, auth_obj = build_dld_token_request(emirates_id, bearer_token)
    response = request_with_bearer("post", url, headers=headers)
    return {
        "status_code": response.status_code,
        "response": response_payload(response),
        "url": url,
        "headers": headers,
        "payload": auth_obj
    }


def get_dld_token(emirates_id, bearer_token):
    result = get_dld_token_result(emirates_id, bearer_token)
    if result.get("status_code", 500) >= 400:
        raise ApiRequestError(f"DLD token lookup failed HTTP {result.get('status_code')}: {compact_payload(result.get('response'))}", "GET_DLD_TOKEN", result.get("url"), result.get("headers"), result=result)
    response_data = result.get("response")
    token = response_data.get("token") if isinstance(response_data, dict) else None
    if not token:
        raise ApiRequestError(f"DLD token lookup returned no token for Emirates ID {emirates_id}: {compact_payload(response_data)}", "GET_DLD_TOKEN", result.get("url"), result.get("headers"), result=result)
    return token

# ================= API3 =================
def get_properties(dld_token, bearer_token, property_type="vacant", type=3):
    url = f"{DLD_BASE_URL}/generalservices/1.0.0/owner-assets/{property_type}/{type}"

    headers = {
        "Token": dld_token,
        "consumer-id": CONSUMER_ID,
        "Authorization": f"Bearer {bearer_token}"
    }

    response = request_with_bearer("get", url, headers=headers)
    data = response_payload(response)
    if response.status_code != 200:
        raise ApiRequestError(f"API3 get properties failed HTTP {response.status_code}: {compact_payload(data)}", "API3_GET_PROPERTIES", url, headers)

    response_data = data.get("Response") if isinstance(data, dict) else None
    properties = response_data.get("Properties") if isinstance(response_data, dict) else None
    if not isinstance(properties, list):
        raise ApiRequestError(f"API3 get properties returned unexpected payload for {property_type}/{type}: {compact_payload(data)}", "API3_GET_PROPERTIES", url, headers)

    return properties


def annotate_owner_asset_source(properties, list_type, property_type_id):
    for prop in properties or []:
        if isinstance(prop, dict):
            prop["_owner_asset_list_type"] = list_type
            prop["_owner_asset_property_type_id"] = property_type_id
    return properties


def owner_asset_detail_list_type_candidates(source_list_type):
    candidates = [source_list_type, "leased", "rented", "vacant", "owned"]
    ordered = []
    for candidate in candidates:
        candidate = str(candidate or "").strip().lower()
        if candidate and candidate not in ordered:
            ordered.append(candidate)
    return ordered


def build_owner_asset_property_detail_request(list_type, property_type_id, property_row_value, dld_token, bearer_token):
    url = f"{DLD_BASE_URL}/generalservices/1.0.0/owner-assets/{list_type}/{property_type_id}/{property_row_value}"
    headers = {
        "Token": dld_token,
        "consumer-id": CONSUMER_ID,
        "Authorization": f"Bearer {bearer_token}"
    }
    return url, headers


def property_detail_result_succeeded(result):
    if not api_result_succeeded(result):
        return False
    response = result.get("response") if isinstance(result, dict) else None
    return isinstance((response or {}).get("Response"), dict)


def get_owner_asset_property_detail_result(list_type, property_type_id, property_row_value, dld_token, bearer_token):
    attempts = []
    for detail_list_type in owner_asset_detail_list_type_candidates(list_type):
        url, headers = build_owner_asset_property_detail_request(detail_list_type, property_type_id, property_row_value, dld_token, bearer_token)
        try:
            response = request_with_bearer("get", url, headers=headers)
            sent_headers = final_request_auth_headers(response, headers)
            result = {
                "status_code": response.status_code,
                "response": response_payload(response),
                "url": url,
                "headers": sent_headers,
                "owner_asset_list_type": detail_list_type,
                "source_owner_asset_list_type": list_type,
                "property_type_id": property_type_id,
                "property_row_value": property_row_value
            }
        except Exception as exc:
            result = {
                "status_code": None,
                "response": None,
                "url": url,
                "headers": headers,
                "owner_asset_list_type": detail_list_type,
                "source_owner_asset_list_type": list_type,
                "property_type_id": property_type_id,
                "property_row_value": property_row_value,
                "error": str(exc)
            }
        attempts.append(result)
        if property_detail_result_succeeded(result):
            return attach_attempts_snapshot(result, attempts)

    final_result = attempts[-1] if attempts else {"status_code": None, "response": None}
    return attach_attempts_snapshot(final_result, attempts)

# ================= API4 =================
def build_validate_property_request(row_value, dld_token, bearer_token):
    url = f"{DLD_BASE_URL}/dxbnwejaricontracts/1.0.0/properties/{row_value}/validate"
    headers = {
        "Token": dld_token,
        "consumer-id": CONSUMER_ID,
        "Authorization": f"Bearer {bearer_token}"
    }
    return url, headers

def validate_property(row_value, dld_token, bearer_token):
    url, headers = build_validate_property_request(row_value, dld_token, bearer_token)

    response = request_with_bearer("get", url, headers=headers)
    if response.status_code != 200:
        raise ApiRequestError(f"API4 validate failed HTTP {response.status_code}: {compact_payload(response_payload(response))}", "API4_VALIDATE_PROPERTY", url, headers)

    data = response_payload(response)
    response_data = data.get("Response") if isinstance(data, dict) else None
    ejari_property_ids = response_data.get("EjariPropertyIDs") if isinstance(response_data, dict) else None

    if not ejari_property_ids:
        raise ApiRequestError(f"API4 validate returned no EjariPropertyIDs for property_row_value={row_value}: {compact_payload(data)}", "API4_VALIDATE_PROPERTY", url, headers)

    return ejari_property_ids[0]

# ================= TENANT DETAILS =================
TENANT_CREATE_FIELDS = [
    "IsPrimary",
    "TenantName",
    "Email",
    "TenantNumber",
    "Pobox",
    "Gender",
    "Mobile",
    "PassportNo",
    "NationalityId",
    "PassportIssuePlace",
    "PassportIssueDate",
    "PassportExpiryDate",
    "EmiratesId",
    "ResidenceVisaNumber",
    "VisaStartDate",
    "VisaExpiryDate",
    "DOB"
]

def get_tenant_person(tenant_emirates_id, tenant_dob, dld_token, bearer_token):
    tenant_emirates_id = str(tenant_emirates_id).strip()
    tenant_dob = str(tenant_dob).strip()
    url = f"{DLD_BASE_URL}/dxbnwejaricontracts/1.0.0/contracts/gettenantperson/{tenant_emirates_id}/{tenant_dob}"

    headers = {
        "accept": "text/plain",
        "Token": dld_token,
        "consumer-id": CONSUMER_ID,
        "Authorization": f"Bearer {bearer_token}"
    }

    response = request_with_bearer("get", url, headers=headers)
    data = response_payload(response)
    if response.status_code != 200:
        raise ApiRequestError(f"Tenant details lookup failed HTTP {response.status_code}: {compact_payload(data)}", "GET_TENANT_PERSON", url, headers)

    errors = data.get("ValidationErrorsList") if isinstance(data, dict) else None
    first_error = errors[0] if errors else None
    first_error_number = first_error.get("ErrorNumber") if isinstance(first_error, dict) else None
    if first_error and first_error_number != 0:
        raise ApiRequestError(f"Tenant details lookup failed validation: {compact_payload(data)}", "GET_TENANT_PERSON", url, headers)

    response_data = data.get("Response") if isinstance(data, dict) else None
    tenant = response_data.get("Tenant") if isinstance(response_data, dict) else None
    if not isinstance(tenant, dict):
        raise ApiRequestError(f"Tenant details lookup returned no Tenant object: {compact_payload(data)}", "GET_TENANT_PERSON", url, headers)

    return tenant

def build_create_contract_tenant(tenant):
    create_tenant = {
        field: tenant[field]
        for field in TENANT_CREATE_FIELDS
        if field in tenant
    }
    create_tenant["IsPrimary"] = tenant.get("IsPrimary", True)
    return create_tenant

# ================= API5 =================
def build_create_contract_request(ejari_id, dld_token, bearer_token, tenant):
    url = f"{DLD_PROXY_URL}/ejariservices/contract/create"

    headers = {
        "accept": "text/plain",
        "Content-Type": "application/json-patch+json",
        "consumer-id": CONSUMER_ID,
        "Token": dld_token,
        "Authorization": f"Bearer {bearer_token}"
    }

    payload = {
        "ContractStartDate": "2026-01-15T00:00:00Z",
        "ContractEndDate": "2028-01-14T00:00:00Z",
        "GraceStartDate": "2027-12-25T00:00:00Z",
        "GraceEndDate": "2028-01-14T00:00:00Z",
        "ContractValue": 74000.0,
        "DiscountValue": 0.0,
        "DiscountTypeId": 0.0,
        "ContractActualValue": 69000.0,
        "ContractAnnualValue": 38196.58,
        "ContractActualAnnualValue": 35576.92,
        "AssociatedContractAnnualCosts": [
          {
            "StartDate": "2026-01-15T00:00:00Z",
            "EndDate": "2027-01-14T00:00:00Z",
            "Value": 34000.0,
            "DiscountValue": 1000.0,
            "ActualValue": 33000.0,
            "DiscountTypeId": 2.0,
            "AnnualValue": 34000.0,
            "YearRatio": 1.0
          },
          {
            "StartDate": "2027-01-15T00:00:00Z",
            "EndDate": "2027-12-24T00:00:00Z",
            "Value": 40000.0,
            "DiscountValue": 10.0,
            "ActualValue": 36000.0,
            "DiscountTypeId": 3.0,
            "AnnualValue": 42393.1623931624,
            "YearRatio": 0.943548387096774
          }
        ],
        "NoOfPayments": 20,
        "Payments": [
          {
            "Amount": 3450.0,
            "InstallmentDueDate": "2026-01-15",
            "PaymentTypeId": 2
          },
          {
            "Amount": 3450.0,
            "ChequeNumber": "12345",
            "InstallmentDueDate": "2026-01-15",
            "PaymentTypeId": 1,
            "BankName": "18,165,142"
          },
          {
            "Amount": 3450.0,
            "InstallmentDueDate": "2026-01-15",
            "PaymentTypeId": 2
          },
          {
            "Amount": 3450.0,
            "ChequeNumber": "11111",
            "InstallmentDueDate": "2026-01-15",
            "PaymentTypeId": 1,
            "BankName": "133"
          },
          {
            "Amount": 3450.0,
            "InstallmentDueDate": "2026-01-15",
            "PaymentTypeId": 2
          },
          {
            "Amount": 3450.0,
            "ChequeNumber": "23337",
            "InstallmentDueDate": "2026-01-15",
            "PaymentTypeId": 1,
            "BankName": "89"
          },
          {
            "Amount": 3450.0,
            "ChequeNumber": "6272727",
            "InstallmentDueDate": "2026-01-15",
            "PaymentTypeId": 1,
            "BankName": "345,633,046"
          },
          {
            "Amount": 3450.0,
            "InstallmentDueDate": "2026-01-15",
            "PaymentTypeId": 2
          },
          {
            "Amount": 3450.0,
            "ChequeNumber": "16278",
            "InstallmentDueDate": "2026-01-15",
            "PaymentTypeId": 1,
            "BankName": "42"
          },
          {
            "Amount": 3450.0,
            "ChequeNumber": "45166",
            "InstallmentDueDate": "2026-01-15",
            "PaymentTypeId": 1,
            "BankName": "42"
          },
          {
            "Amount": 3450.0,
            "ChequeNumber": "555",
            "InstallmentDueDate": "2026-01-15",
            "PaymentTypeId": 1,
            "BankName": "6"
          },
          {
            "Amount": 3450.0,
            "ChequeNumber": "16272772",
            "InstallmentDueDate": "2026-01-15",
            "PaymentTypeId": 1,
            "BankName": "482,701,363"
          },
          {
            "Amount": 3450.0,
            "ChequeNumber": "16727",
            "InstallmentDueDate": "2026-01-15",
            "PaymentTypeId": 1,
            "BankName": "6"
          },
          {
            "Amount": 3450.0,
            "InstallmentDueDate": "2026-01-15",
            "PaymentTypeId": 2
          },
          {
            "Amount": 3450.0,
            "InstallmentDueDate": "2026-01-15",
            "PaymentTypeId": 2
          },
          {
            "Amount": 3450.0,
            "InstallmentDueDate": "2026-01-15",
            "PaymentTypeId": 2
          },
          {
            "Amount": 3450.0,
            "ChequeNumber": "2662626",
            "InstallmentDueDate": "2026-01-15",
            "PaymentTypeId": 1,
            "BankName": "6"
          },
          {
            "Amount": 3450.0,
            "ChequeNumber": "16236",
            "InstallmentDueDate": "2026-01-15",
            "PaymentTypeId": 1,
            "BankName": "6"
          },
          {
            "Amount": 3450.0,
            "ChequeNumber": "1111",
            "InstallmentDueDate": "2026-01-15",
            "PaymentTypeId": 1,
            "BankName": "null"
          },
          {
            "Amount": 3450.0,
            "InstallmentDueDate": "2026-01-15",
            "PaymentTypeId": 2
          }
        ],
        "NoOfCoocupants": 0,
        "ContractValueType": 2,
        "AdditionalTerms": [
          {
            "EnglishTerm": "do jkk nkkkd dndjjss njdjdjdjdsnsjsjsjsjjsnzsjjsjsjsjzjzkzz z sjjsjs sjsjsjssns sn shhshs snsjsj sjsjjs sbjsjs sjsjsjs jsjsjsjs jsjsjsjsj shsjjsjsjs jsjsjsj jsjsjsjsjs jsjsjsjsjsjsj shsjsjsjjs",
            "ArabicTerm": "يتنيين ينينيني ينيننيني ينينينينني سنينيننسني ستسننينيني تسنينينين نينينينني ستنينينيني تسنيننيني تسنينينين تينينينث نينينينني تسنثنيني تينين"
          }
        ],
        "Tenants": [
          build_create_contract_tenant(tenant)
        ],
        "SecurityDeposit": 5000.0,
        "EjariPropertyIds": [
          ejari_id
        ]
      }

    return url, headers, payload


def create_contract(ejari_id, dld_token, bearer_token, tenant):
    url, headers, payload = build_create_contract_request(ejari_id, dld_token, bearer_token, tenant)
    response = request_with_bearer("post", url, headers=headers, json=payload)
    sent_headers = final_request_auth_headers(response, headers)

    # Important: API may return text/plain
    try:
      resp_json = response.json()
    except:
      resp_json = response.text

    return {
      "status_code": response.status_code,
      "response": resp_json,
      "url": url,
      "headers": sent_headers,
      "payload": payload
    }

def get_created_contract_response_data(api5_response):
    response_data = api5_response.get("Response") if isinstance(api5_response, dict) else None
    return response_data if isinstance(response_data, dict) else {}


def get_created_contract_row_value(api5_response):
    contract_row_value = get_created_contract_response_data(api5_response).get("DataRowValue")
    if not contract_row_value:
        raise ApiRequestError(f"API5 create response did not include Response.DataRowValue: {compact_payload(api5_response)}", "API5_CREATE_CONTRACT")
    return contract_row_value


def get_created_contract_number(api5_response):
    return get_created_contract_response_data(api5_response).get("ContractNumber")


def build_download_tenancy_contract_request(contract_row_value, dld_token, bearer_token):
    url = f"{DLD_PROXY_URL}/ejariservices/contracts/{contract_row_value}/downloadTenancyContract"
    headers = {"accept": "text/plain", "consumer-id": CONSUMER_ID, "Token": dld_token, "Authorization": f"Bearer {bearer_token}"}
    return url, headers


def download_tenancy_contract(contract_row_value, dld_token, bearer_token):
    url, headers = build_download_tenancy_contract_request(contract_row_value, dld_token, bearer_token)
    response = request_with_bearer("get", url, headers=headers)
    sent_headers = final_request_auth_headers(response, headers)
    return {"status_code": response.status_code, "response": response_payload(response), "url": url, "headers": sent_headers}


def extract_contract_file(download_response):
    candidates = []
    file_keys = (
        "ContractFile", "contractFile", "contract_file",
        "FileContent", "fileContent", "file_content",
        "Document", "document",
        "Pdf", "pdf",
        "ReportData", "reportData", "report_data"
    )
    if isinstance(download_response, dict):
        for key in file_keys:
            candidates.append(download_response.get(key))
        response_data = download_response.get("Response")
        if isinstance(response_data, dict):
            for key in file_keys:
                candidates.append(response_data.get(key))
        elif isinstance(response_data, str):
            candidates.append(response_data)
    elif isinstance(download_response, str):
        candidates.append(download_response)
    for candidate in candidates:
        if isinstance(candidate, str) and candidate.strip():
            return candidate.strip()
    return None


def download_result_has_contract_file(result):
    return api_result_succeeded(result) and bool(extract_contract_file(result.get("response") if isinstance(result, dict) else None))


def build_save_signed_tenancy_contract_request(contract_row_value, dld_token, bearer_token, contract_file, signed_party, signed_party_type):
    if not signed_party:
        raise ValueError("SignedParty is required for save signed tenancy contract.")
    if signed_party_type not in (1, 5):
        raise ValueError("SignedPartyType must be 1 for owner or 5 for tenant.")
    url = f"{DLD_PROXY_URL}/ejariservices/contracts/{contract_row_value}/savesignedtenancycontract"
    headers = {"accept": "text/plain", "Content-Type": "application/json-patch+json", "consumer-id": CONSUMER_ID, "Token": dld_token, "Authorization": f"Bearer {bearer_token}"}
    payload = {
        "ContractFile": contract_file,
        "SignedParty": signed_party,
        "SignedPartyType": signed_party_type,
        "DataRowValue": contract_row_value
    }
    return url, headers, payload


def save_signed_tenancy_contract(contract_row_value, dld_token, bearer_token, contract_file, signed_party, signed_party_type):
    url, headers, payload = build_save_signed_tenancy_contract_request(contract_row_value, dld_token, bearer_token, contract_file, signed_party, signed_party_type)
    response = request_with_bearer("post", url, headers=headers, json=payload)
    sent_headers = final_request_auth_headers(response, headers)
    return {"status_code": response.status_code, "response": response_payload(response), "url": url, "headers": sent_headers, "payload": payload}

def cancel_contract(contract_row_value, dld_token, bearer_token):
    url = f"{DLD_BASE_URL}/dxbnwejaricontracts/1.0.0/contracts/{contract_row_value}/cancel"

    headers = {
        "accept": "text/plain",
        "Content-Type": "application/json-patch+json",
        "Token": dld_token,
        "consumer-id": CONSUMER_ID,
        "Authorization": f"Bearer {bearer_token}"
    }

    response = request_with_bearer("get", url, headers=headers)
    result = {"status_code": response.status_code, "response": response_payload(response), "url": url, "headers": headers}

    print(f"\nCancel Contract API for: {contract_row_value}")
    print(f"Status: {response.status_code}")
    response_summary = "No content" if response.status_code == 204 else compact_payload(redact_sensitive_values(result.get("response")), 500)
    print(f"Response summary: {response_summary}")

    if response.status_code == 204:
        return result
    if response.status_code != 200 or not api_result_succeeded(result):
        raise Exception(f"Cancel failed: {result_error_summary(result)}")

    return result

def extract_contract_row_value_from_creation_record(prop_data):
    if not isinstance(prop_data, dict):
        return None
    direct_value = prop_data.get("contract_row_value")
    if direct_value:
        return direct_value

    api5_response = prop_data.get("api5_response")
    if isinstance(api5_response, dict):
        response_data = api5_response.get("Response")
        if isinstance(response_data, dict) and response_data.get("DataRowValue"):
            return response_data.get("DataRowValue")

    for step in prop_data.get("step_results") or []:
        if not isinstance(step, dict) or step.get("api_stage") != "API5_CREATE_CONTRACT":
            continue
        result = step.get("result") or {}
        response_data = ((result.get("response") or {}).get("Response") if isinstance(result, dict) else None)
        if isinstance(response_data, dict) and response_data.get("DataRowValue"):
            return response_data.get("DataRowValue")

    return None


def extract_contract_number_from_creation_record(prop_data):
    if not isinstance(prop_data, dict):
        return None
    direct_value = prop_data.get("contract_number")
    if direct_value:
        return direct_value

    api5_response = prop_data.get("api5_response")
    if isinstance(api5_response, dict):
        response_data = api5_response.get("Response")
        if isinstance(response_data, dict) and response_data.get("ContractNumber"):
            return response_data.get("ContractNumber")

    for step in prop_data.get("step_results") or []:
        if not isinstance(step, dict) or step.get("api_stage") != "API5_CREATE_CONTRACT":
            continue
        result = step.get("result") or {}
        response_data = ((result.get("response") or {}).get("Response") if isinstance(result, dict) else None)
        if isinstance(response_data, dict) and response_data.get("ContractNumber"):
            return response_data.get("ContractNumber")

    return None


def get_contract_records_from_progress_file(progress, emirates_id):
    records = []
    seen_contracts = set()
    emirates_data = progress.get(emirates_id, {}) if isinstance(progress, dict) else {}
    record_sets = [
        ("success", emirates_data.get("ejari_creation_success", {})),
        ("failure", emirates_data.get("ejari_creation_failure", {})),
    ]

    for source_status, properties in record_sets:
        for property_key, prop_data in (properties or {}).items():
            try:
                contract_row_value = extract_contract_row_value_from_creation_record(prop_data)
                if not contract_row_value or contract_row_value in seen_contracts:
                    continue
                seen_contracts.add(contract_row_value)
                records.append({
                    "source": source_status,
                    "property_key": property_key,
                    "contract_row_value": contract_row_value,
                    "contract_number": extract_contract_number_from_creation_record(prop_data),
                    "title": prop_data.get("title"),
                    "property_id": prop_data.get("property_id"),
                    "property_row_value": prop_data.get("property_row_value") or property_key,
                })
            except Exception:
                continue

    return records


def get_contract_row_values_from_progress_file(progress, emirates_id):
    return [record["contract_row_value"] for record in get_contract_records_from_progress_file(progress, emirates_id)]


##############################################################
####################### GET CONTRACTS ########################
##############################################################
def contract_list_urls():
    return [
        ("actual", f"{DLD_BASE_URL}/dxbnwejaricontracts/1.0.0/properties/getcontracts"),
        ("proxy", f"{DLD_PROXY_URL}/ejariservices/properties/getcontracts"),
    ]


def contract_details_urls(contract_row_value):
    return [
        ("actual", f"{DLD_BASE_URL}/dxbnwejaricontracts/1.0.0/contracts/{contract_row_value}"),
        ("proxy", f"{DLD_PROXY_URL}/ejariservices/contracts/{contract_row_value}"),
    ]


def attach_attempts_snapshot(result, attempts):
    result_copy = dict(result or {})
    result_copy["attempts"] = [dict(attempt) for attempt in (attempts or [])]
    return result_copy


def get_contracts_list_result(dld_token, bearer_token):
    headers = {
        "accept": "text/plain",
        "Token": dld_token,
        "consumer-id": CONSUMER_ID,
        "Authorization": f"Bearer {bearer_token}"
    }
    attempts = []
    for endpoint_name, url in contract_list_urls():
        try:
            response = request_with_bearer("get", url, headers=headers)
            result = {"status_code": response.status_code, "response": response_payload(response), "url": url, "headers": headers, "endpoint_name": endpoint_name}
        except Exception as exc:
            result = {"status_code": None, "response": None, "url": url, "headers": headers, "endpoint_name": endpoint_name, "error": str(exc)}
        attempts.append(result)
        if api_result_succeeded(result):
            return attach_attempts_snapshot(result, attempts)
    final_result = attempts[-1] if attempts else {"status_code": None, "response": None}
    return attach_attempts_snapshot(final_result, attempts)


def get_contracts_list(dld_token, bearer_token):
    result = get_contracts_list_result(dld_token, bearer_token)
    if not api_result_succeeded(result):
        raise Exception(f"Get contracts failed: {result_error_summary(result)}")
    return result["response"]

####################################################################
def contract_details_result_succeeded(result):
    if not api_result_succeeded(result):
        return False
    response = result.get("response") if isinstance(result, dict) else None
    contract_details = ((response or {}).get("Response") or {}).get("ContractDetails")
    return isinstance(contract_details, dict)


def get_contract_details_result(contract_row_value, dld_token, bearer_token):
    headers = {
        "accept": "text/plain",
        "Content-Type": "application/json-patch+json",
        "Token": dld_token,
        "consumer-id": CONSUMER_ID,
        "Authorization": f"Bearer {bearer_token}"
    }
    attempts = []
    for endpoint_name, url in contract_details_urls(contract_row_value):
        try:
            response = request_with_bearer("get", url, headers=headers)
            sent_headers = final_request_auth_headers(response, headers)
            result = {"status_code": response.status_code, "response": response_payload(response), "url": url, "headers": sent_headers, "endpoint_name": endpoint_name}
        except Exception as exc:
            result = {"status_code": None, "response": None, "url": url, "headers": headers, "endpoint_name": endpoint_name, "error": str(exc)}
        attempts.append(result)
        if contract_details_result_succeeded(result):
            return attach_attempts_snapshot(result, attempts)
    final_result = attempts[-1] if attempts else {"status_code": None, "response": None}
    return attach_attempts_snapshot(final_result, attempts)


def get_contract_details(contract_row_value, dld_token, bearer_token):
    result = get_contract_details_result(contract_row_value, dld_token, bearer_token)
    if not contract_details_result_succeeded(result):
        raise Exception(f"Contract details failed: {result_error_summary(result)}")
    return result["response"]

def text_name(value):
    if isinstance(value, dict):
        return value.get("EnglishName") or value.get("ArabicName") or value.get("Value") or value.get("Name")
    return value


def contract_title(contract):
    if not isinstance(contract, dict):
        return None
    return text_name(contract.get("Title")) or text_name(contract.get("PropertyNumber")) or contract.get("ContractNumber") or contract_row_value_from_contract(contract)


def contract_status_name(contract):
    if not isinstance(contract, dict):
        return None
    return text_name(contract.get("ContractStatus"))


def contract_status_identity(contract):
    status = contract.get("ContractStatus") if isinstance(contract, dict) else None
    if isinstance(status, dict):
        return status.get("Identity") or status.get("Value")
    return None


def normalize_status_name(value):
    return str(value or "").strip().lower().replace("_", " ").replace("-", " ")


def is_pending_contract(contract):
    status_name = normalize_status_name(contract_status_name(contract))
    status_identity = str(contract_status_identity(contract) or "").strip()
    return status_name == "pending" or status_identity == "1"


def contract_row_value_from_contract(contract):
    if not isinstance(contract, dict):
        return None
    return (
        contract.get("AssociatedContractRowValue")
        or contract.get("ContractRowValue")
        or contract.get("DataRowValue")
        or contract.get("contract_row_value")
    )


def consolidate_unique_contracts_from_contracts_response(contracts_response, only_pending=False):
    response_data = contracts_response.get("Response") if isinstance(contracts_response, dict) else {}
    unique_contracts = {}
    for source_name in ("OwnerContracts", "TenantContracts"):
        for contract in response_data.get(source_name, []) or []:
            if not isinstance(contract, dict):
                continue
            if only_pending and not is_pending_contract(contract):
                continue
            contract_row_value = contract_row_value_from_contract(contract)
            if not contract_row_value:
                continue
            if contract_row_value not in unique_contracts:
                consolidated = dict(contract)
                consolidated["source_lists"] = [source_name]
                consolidated["is_owner_contract_list_item"] = source_name == "OwnerContracts"
                consolidated["is_tenant_contract_list_item"] = source_name == "TenantContracts"
                unique_contracts[contract_row_value] = consolidated
            else:
                existing = unique_contracts[contract_row_value]
                existing.setdefault("source_lists", [])
                if source_name not in existing["source_lists"]:
                    existing["source_lists"].append(source_name)
                if source_name == "OwnerContracts":
                    existing["is_owner_contract_list_item"] = True
                if source_name == "TenantContracts":
                    existing["is_tenant_contract_list_item"] = True
    return list(unique_contracts.values())


def parse_optional_bool(value):
    if isinstance(value, bool):
        return value
    if isinstance(value, int) and value in (0, 1):
        return bool(value)
    if isinstance(value, str):
        normalized = value.strip().lower()
        if normalized in {"true", "1", "yes", "y", "signed"}:
            return True
        if normalized in {"false", "0", "no", "n", "unsigned", "not signed"}:
            return False
    return None


def find_bool_by_key(payload, key_names):
    normalized_keys = {key.lower() for key in key_names}
    if isinstance(payload, dict):
        for key, value in payload.items():
            if str(key).lower() in normalized_keys:
                parsed_value = parse_optional_bool(value)
                if parsed_value is not None:
                    return parsed_value
        for value in payload.values():
            parsed_value = find_bool_by_key(value, key_names)
            if parsed_value is not None:
                return parsed_value
    elif isinstance(payload, list):
        for item in payload:
            parsed_value = find_bool_by_key(item, key_names)
            if parsed_value is not None:
                return parsed_value
    return None


OWNER_SIGNED_KEYS = (
    "IsContractSignedByOwner", "isContractSignedByOwner",
    "contractSignedByOwner", "ContractSignedByOwner",
    "isSignedByOwner", "IsSignedByOwner",
    "ownerSigned", "OwnerSigned",
    "signedByOwner", "SignedByOwner"
)
TENANT_SIGNED_KEYS = (
    "IsContractSignedByTenant", "isContractSignedByTenant",
    "contractSignedByTenant", "ContractSignedByTenant",
    "isSignedByTenant", "IsSignedByTenant",
    "tenantSigned", "TenantSigned",
    "signedByTenant", "SignedByTenant"
)


def contract_details_payload(contract_details_response):
    if not isinstance(contract_details_response, dict):
        return {}
    response_data = contract_details_response.get("Response") or {}
    return response_data.get("ContractDetails") or {}


def contract_signed_flag(contract, contract_details_response, party):
    key_names = OWNER_SIGNED_KEYS if party == "owner" else TENANT_SIGNED_KEYS
    value = find_bool_by_key(contract, key_names)
    if value is not None:
        return value
    return find_bool_by_key(contract_details_payload(contract_details_response), key_names)


def contract_tenant_emirates_id(contract, contract_details_response):
    if isinstance(contract, dict):
        for key in ("TenantEmiratesId", "TenantEmiratesID", "tenant_emirates_id", "EmiratesId"):
            value = contract.get(key)
            if value:
                return str(value)
    details = contract_details_payload(contract_details_response)
    tenants = details.get("Tenants") or []
    if isinstance(tenants, list):
        for tenant in tenants:
            if isinstance(tenant, dict) and tenant.get("EmiratesId"):
                return str(tenant.get("EmiratesId"))
    return None



def contract_owner_participant_id(contract_details_response):
    details = contract_details_payload(contract_details_response)
    return details.get("OwnerParticipantID") or details.get("OwnerParticipantId") or details.get("ownerParticipantId")


def primary_contract_tenant(contract_details_response):
    details = contract_details_payload(contract_details_response)
    tenants = details.get("Tenants") or []
    if not isinstance(tenants, list):
        return {}
    for tenant in tenants:
        if isinstance(tenant, dict) and tenant.get("IsPrimary"):
            return tenant
    for tenant in tenants:
        if isinstance(tenant, dict):
            return tenant
    return {}


def contract_tenant_id(contract_details_response):
    tenant = primary_contract_tenant(contract_details_response)
    return tenant.get("TenantID") or tenant.get("TenantId") or tenant.get("tenantId")

def contract_number_from_contract(contract, contract_details_response=None):
    if isinstance(contract, dict) and contract.get("ContractNumber"):
        return contract.get("ContractNumber")
    details = contract_details_payload(contract_details_response)
    return details.get("ContractNumber")


########################################################################################
def require_dewa_config():
    required_values = {
        "DEWA_BASE_URL": DEWA_BASE_URL,
        "DEWA_CLIENT_ID": DEWA_CLIENT_ID,
        "DEWA_CLIENT_SECRET": DEWA_CLIENT_SECRET,
        "DEWA_VENDOR_ID": DEWA_VENDOR_ID,
    }
    missing = [name for name, value in required_values.items() if not value]
    if missing:
        raise RuntimeError(f"Missing DEWA config values: {', '.join(missing)}")


########################################################################################
def premise_status_check(contract_number, premise_no, bearer_token):
    require_dewa_config()
    url = f"{DEWA_BASE_URL}/moveinprocess/rest/1.0.0/PremiseStatus_Check"

    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {bearer_token}"
    }

    payload = {
        "MT_PremiseStatusCheck_Req": {
            "LangKey": "EN",
            "Vendor": "DLD",
            "Item": {
                "EJARINumber": contract_number,
                "PremiseNo": premise_no,
                "Input1": "",
                "Input2": "",
                "Input3": "",
                "Input4": "",
                "AddnlInput": ""
            }
        }
    }

    response = request_with_bearer("post", url, headers=headers, json=payload)

    if response.status_code != 200:
        raise Exception(f"Premise check failed HTTP {response.status_code}: {compact_payload(response_payload(response), 500)}")

    return response.json()


########################################################################################
def get_dewa_token(bearer_token):
    require_dewa_config()
    url = f"{DEWA_BASE_URL}/smartpaymenttoken/1.0.0/ddaservices/accesstoken?grant_type=client_credentials"

    headers = {
        "Content-Type": "application/x-www-form-urlencoded",
        "Authorization": f"Bearer {bearer_token}"
    }

    data = {
        "client_id": DEWA_CLIENT_ID,
        "client_secret": DEWA_CLIENT_SECRET
    }

    response = request_with_bearer("post", url, headers=headers, data=data)

    if response.status_code != 200:
        raise Exception(f"DEWA token failed HTTP {response.status_code}: {compact_payload(response_payload(response), 500)}")

    return response.json()["access_token"]


########################################################################################
def dewa_enquiry(contract_act_number, dewa_token, bearer_token):
    require_dewa_config()
    url = f"{DEWA_BASE_URL}/smartpaymentbilling/1.0.0/ddaservices/enquiry/{contract_act_number}?channel=WEB"

    headers = {
        "x-nv-header": dewa_token,
        "vendorid": DEWA_VENDOR_ID,
        "Authorization": f"Bearer {bearer_token}"
    }

    response = request_with_bearer("get", url, headers=headers)

    if response.status_code != 200:
        raise Exception(f"DEWA enquiry failed HTTP {response.status_code}: {compact_payload(response_payload(response), 500)}")

    return response.json()


## Contract Cancellation

Cancels pending/created contracts from `progress.json` first. Optionally, it can fetch pending contracts from the API contract list and cancel those as well.


In [ ]:
# @title Contract cancellation
####################################
###### CANCEL CONTRACTS ############
####################################
def cancel_contracts_in_progress_file(emirates_id):
    if not confirm_process_emirates_id("contract cancellation", emirates_id):
        return

    print(f"\nProcessing cancellation for Emirates ID: {emirates_id}")
    progress = load_progress()

    bearer_token = get_active_bearer_token(force_refresh=True)
    print(f"Generated fresh iPaaS bearer token for Emirates ID {emirates_id}")
    dld_token = get_dld_token(emirates_id, bearer_token)

    progress_contracts = get_contract_records_from_progress_file(progress, emirates_id)
    unique_contract_ids = {record["contract_row_value"] for record in progress_contracts if record.get("contract_row_value")}
    processed_contracts = set()
    total_count = len(progress_contracts)
    processed_count = 0
    success_count = 0
    failure_count = 0

    print(f"\nPending/created contracts found in progress.json: {len(progress_contracts)}")
    for contract_record in progress_contracts:
        contract_id = contract_record.get("contract_row_value")
        if not contract_id:
            failure_count += 1
            print(f"ERROR cancelling progress contract: missing contract_row_value in {contract_record}")
            continue
        processed_contracts.add(contract_id)
        processed_count += 1
        try:
            print(f"Cancelling progress contract {contract_record.get('contract_number') or ''} | {contract_id} | {contract_record.get('title') or ''}")
            cancel_contract(contract_id, dld_token, bearer_token)
            success_count += 1
        except Exception as e:
            failure_count += 1
            print(f"ERROR cancelling {contract_id}: {e}")

    api_cancel_requested = ask_yes_no("Fetch pending contracts from API contract list and cancel them too?", default=False)
    if api_cancel_requested:
        try:
            contracts_response = get_contracts_list(dld_token, bearer_token)
        except Exception as e:
            failure_count += 1
            print(f"ERROR fetching contracts list for cancellation: {e}")
            print_processing_summary("contract cancellation", emirates_id, total_count, len(unique_contract_ids), processed_count, success_count, failure_count)
            return

        response_data = contracts_response.get("Response") if isinstance(contracts_response, dict) else {}
        api_total_count = sum(
            1
            for source_name in ("OwnerContracts", "TenantContracts")
            for contract in (response_data.get(source_name, []) or [])
            if isinstance(contract, dict) and is_pending_contract(contract)
        )
        total_count += api_total_count
        pending_contracts = consolidate_unique_contracts_from_contracts_response(contracts_response, only_pending=True)
        for contract in pending_contracts:
            contract_id = contract_row_value_from_contract(contract)
            if contract_id:
                unique_contract_ids.add(contract_id)
        print(f"Pending contract list items from API: {api_total_count}")
        print(f"Pending unique contracts found from API: {len(pending_contracts)}")

        for contract in pending_contracts:
            contract_id = contract_row_value_from_contract(contract)
            if not contract_id:
                failure_count += 1
                print("ERROR cancelling API pending contract: missing AssociatedContractRowValue/DataRowValue")
                continue
            if contract_id in processed_contracts:
                print(f"Skipping API pending contract already processed from progress: {contract_id}")
                continue
            processed_contracts.add(contract_id)
            processed_count += 1
            try:
                print(f"Cancelling API pending contract {contract.get('ContractNumber') or ''} | {contract_id} | {contract_title(contract) or ''}")
                cancel_contract(contract_id, dld_token, bearer_token)
                success_count += 1
            except Exception as e:
                failure_count += 1
                print(f"ERROR cancelling API pending contract {contract_id}: {e}")

    print_processing_summary("contract cancellation", emirates_id, total_count, len(unique_contract_ids), processed_count, success_count, failure_count)

run_check = ask_yes_no("Do you want to run Contract cancellation scripts?", default=False)
if run_check:
    ask_process_all_emirates_ids("contract cancellation")
    for owner_emirates_id in OWNER_EMIRATES_IDS:
        cancel_contracts_in_progress_file(owner_emirates_id)
else:
    print("Skipped contract cancellation scripts")


## Contract Termination

Processes leased contracts for configured Emirates IDs. Actual termination requires an explicit `TERMINATE` confirmation; curl mode only writes request repro files.


In [ ]:
# @title Contract termination
##########################################################################################
####################### TERMINATE LEASED CONTRACTS #######################################
##########################################################################################

TERMINATION_OWNER_EMIRATES_IDS = OWNER_EMIRATES_IDS

TERMINATION_STORAGE_FILE = "progress-termination.json"


def display_name(value):
    if isinstance(value, dict):
        return value.get("EnglishName") or value.get("ArabicName") or compact_payload(value)
    return value


def parse_contract_datetime(value):
    from datetime import timezone
    if not value:
        return None
    text = str(value).strip()
    if text.endswith("Z"):
        text = text[:-1] + "+00:00"
    parsed = datetime.fromisoformat(text)
    if parsed.tzinfo is None:
        parsed = parsed.replace(tzinfo=timezone.utc)
    return parsed.astimezone(timezone.utc)


def format_termination_datetime(value):
    from datetime import timezone
    return value.astimezone(timezone.utc).isoformat(timespec="milliseconds").replace("+00:00", "Z")


def derive_termination_date_from_contract(contract_details):
    from datetime import timezone
    contract_start_date = contract_details.get("ContractStartDate")
    contract_end_date = contract_details.get("ContractEndDate")
    start_dt = parse_contract_datetime(contract_start_date)
    end_dt = parse_contract_datetime(contract_end_date)
    if not start_dt or not end_dt:
        raise ValueError(f"Contract start/end date is missing. start={contract_start_date}, end={contract_end_date}")
    if end_dt < start_dt:
        raise ValueError(f"Contract end date is before start date. start={contract_start_date}, end={contract_end_date}")

    now_dt = datetime.now(timezone.utc)
    if now_dt < start_dt:
        termination_dt = start_dt
        source = "contract_start_date"
    elif now_dt > end_dt:
        termination_dt = end_dt
        source = "contract_end_date"
    else:
        termination_dt = now_dt
        source = "current_utc_time"

    return {
        "termination_date": format_termination_datetime(termination_dt),
        "termination_date_source": source,
        "contract_start_date": contract_start_date,
        "contract_end_date": contract_end_date
    }


def api_payload_errors(payload):
    if not isinstance(payload, dict):
        return [{
            "ErrorNumber": None,
            "ErrorMessageSet": {
                "EnglishName": "API response payload is not a JSON object."
            },
            "ValidationType": "ResponsePayload"
        }]

    errors = []
    response_code = payload.get("ResponseCode")
    if response_code is not None and str(response_code).strip().lower() != "success":
        errors.append({
            "ErrorNumber": None,
            "ErrorMessageSet": {
                "EnglishName": f"ResponseCode={response_code}"
            },
            "ValidationType": "ResponseCode"
        })

    top_level_error_message = payload.get("ErrorMessage")
    if top_level_error_message:
        errors.append({
            "ErrorNumber": None,
            "ErrorMessageSet": {
                "EnglishName": str(top_level_error_message)
            },
            "ValidationType": "ErrorMessage"
        })

    for api_error in payload.get("ValidationErrorsList") or []:
        if not isinstance(api_error, dict):
            errors.append({
                "ErrorNumber": None,
                "ErrorMessageSet": {
                    "EnglishName": str(api_error)
                },
                "ValidationType": "ValidationErrorsList"
            })
            continue

        error_number = api_error.get("ErrorNumber")
        if error_number not in (None, 0, "0"):
            errors.append(api_error)

    return errors


def describe_api_payload_errors(payload):
    messages = []
    for api_error in api_payload_errors(payload):
        messages.append(get_api5_error_message(api_error))
    return " | ".join(messages)


def api_result_succeeded(result):
    if not isinstance(result, dict):
        return False
    status_code = result.get("status_code")
    if status_code is None or status_code >= 400:
        return False
    return not api_payload_errors(result.get("response"))


def approval_result_succeeded(approval_result):
    attempts = approval_result.get("attempts") or [] if isinstance(approval_result, dict) else []
    return any(api_result_succeeded(attempt) for attempt in attempts)


def normalize_termination_progress(progress):
    for emirates_data in progress.values():
        if not isinstance(emirates_data, dict):
            continue
        success_records = emirates_data.setdefault("ejari_termination_success", {})
        failure_records = emirates_data.setdefault("ejari_termination_failure", {})
        emirates_data.setdefault("ejari_termination_curl_only", {})

        for contract_key, record in list(success_records.items()):
            if not isinstance(record, dict):
                continue

            terminate_result = record.get("terminate_result")
            if terminate_result is not None and not api_result_succeeded(terminate_result):
                record["status"] = "TERMINATE_FAILED"
                record["api_stage"] = "TERMINATE_CONTRACT_BY_OWNER"
                record["error"] = "Terminate Contract By Owner returned API errors: " + (describe_api_payload_errors(terminate_result.get("response")) or compact_payload(terminate_result.get("response")))
                failure_records[contract_key] = record
                success_records.pop(contract_key, None)
                continue

            approval_results = record.get("approval_results") or []
            if approval_results and any(not approval_result_succeeded(approval_result) for approval_result in approval_results):
                record["status"] = "TERMINATED_APPROVAL_FAILED"
                record["api_stage"] = "TENANT_APPROVAL_FOR_TERMINATION"
                record["error"] = "Tenant approval for termination returned API errors."
                failure_records[contract_key] = record
                success_records.pop(contract_key, None)

    return progress


def load_termination_progress():
    if os.path.exists(TERMINATION_STORAGE_FILE):
        with open(TERMINATION_STORAGE_FILE, "r", encoding="utf-8") as f:
            return normalize_termination_progress(json.load(f))
    return {}


TERMINATION_FAILURE_REPORT_FIELDS = [
    "emirates_id",
    "property_ids",
    "property_row_values",
    "titles",
    "contract_row_value",
    "contract_number",
    "status",
    "api_stage",
    "failed_step",
    "token_source",
    "tenant_sequence",
    "tenant_emirates_id",
    "http_status_code",
    "attempt",
    "automation_error",
    "api_error",
    "error",
    "api_request_curl_path",
    "api_response_path",
    "terminate_request_curl_path",
    "terminate_response_path",
    "approval_request_curl_path",
    "approval_response_path",
    "failure_json_path",
    "timestamp"
]


def termination_result_report_error(result):
    if not isinstance(result, dict):
        return "Missing API result", ""
    automation_error = result.get("error") or ""
    status_code = result.get("status_code")
    if status_code is not None and status_code >= 400:
        automation_error = (automation_error + "; " if automation_error else "") + f"HTTP {status_code}"
    payload_error = describe_api_payload_errors(result.get("response"))
    formatted_payload = format_failure_api_error(result.get("response"))
    api_error = payload_error
    if formatted_payload and formatted_payload not in {"null", api_error}:
        api_error = (api_error + " | " if api_error else "") + formatted_payload
    return automation_error, api_error


def termination_base_report_row(emirates_id, record):
    return {
        "emirates_id": emirates_id,
        "property_ids": ", ".join(str(value) for value in record.get("property_ids", [])),
        "property_row_values": ", ".join(str(value) for value in record.get("property_row_values", [])),
        "titles": " | ".join(str(value) for value in record.get("titles", [])),
        "contract_row_value": record.get("contract_row_value"),
        "contract_number": record.get("contract_number"),
        "status": record.get("status"),
        "api_stage": record.get("api_stage"),
        "error": record.get("error"),
        "failure_json_path": record.get("failure_json_path"),
        "timestamp": record.get("timestamp")
    }


def build_termination_failure_report(progress):
    rows = []
    for emirates_id, emirates_data in progress.items():
        if not isinstance(emirates_data, dict):
            continue
        failure_records = emirates_data.get("ejari_termination_failure", {})
        for contract_key, record in failure_records.items():
            base_row = termination_base_report_row(emirates_id, record)
            added_step_row = False
            terminate_result = record.get("terminate_result")
            if terminate_result is not None and not api_result_succeeded(terminate_result):
                automation_error, api_error = termination_result_report_error(terminate_result)
                row = dict(base_row)
                row.update({"failed_step": "TERMINATE_CONTRACT_BY_OWNER", "http_status_code": terminate_result.get("status_code"), "attempt": terminate_result.get("attempt"), "automation_error": automation_error, "api_error": api_error, "api_request_curl_path": record.get("terminate_request_curl_path"), "api_response_path": record.get("terminate_response_path"), "terminate_request_curl_path": record.get("terminate_request_curl_path"), "terminate_response_path": record.get("terminate_response_path")})
                rows.append(row)
                added_step_row = True
            for approval_result in record.get("approval_results") or []:
                if approval_result_succeeded(approval_result):
                    continue
                tenant_sequence = approval_result.get("tenant_sequence")
                tenant_emirates_id = approval_result.get("tenant_emirates_id")
                for attempt in approval_result.get("attempts") or []:
                    token_result = attempt.get("token_result")
                    if token_result is not None and not dld_token_result_succeeded(token_result):
                        automation_error, api_error = termination_result_report_error(token_result)
                        row = dict(base_row)
                        row.update({"failed_step": "TENANT_DLD_TOKEN", "token_source": attempt.get("token_source"), "tenant_sequence": tenant_sequence, "tenant_emirates_id": tenant_emirates_id, "http_status_code": token_result.get("status_code"), "attempt": token_result.get("attempt"), "automation_error": attempt.get("error") or automation_error, "api_error": api_error})
                        rows.append(row)
                        added_step_row = True
                        continue
                    legacy_token_failure = attempt.get("api_stage") == "TENANT_DLD_TOKEN" or (attempt.get("token_source") == "tenant" and attempt.get("status_code") is None and bool(attempt.get("error")) and not attempt.get("url"))
                    if legacy_token_failure:
                        row = dict(base_row)
                        row.update({"failed_step": "TENANT_DLD_TOKEN", "token_source": attempt.get("token_source"), "tenant_sequence": tenant_sequence, "tenant_emirates_id": tenant_emirates_id, "http_status_code": attempt.get("status_code"), "attempt": attempt.get("attempt"), "automation_error": attempt.get("error"), "api_error": ""})
                        rows.append(row)
                        added_step_row = True
                        continue
                    if not api_result_succeeded(attempt):
                        automation_error, api_error = termination_result_report_error(attempt)
                        row = dict(base_row)
                        row.update({"failed_step": "TENANT_APPROVAL_FOR_TERMINATION", "token_source": attempt.get("token_source"), "tenant_sequence": tenant_sequence, "tenant_emirates_id": tenant_emirates_id, "http_status_code": attempt.get("status_code"), "attempt": attempt.get("attempt"), "automation_error": automation_error, "api_error": api_error, "api_request_curl_path": attempt.get("api_request_curl_path"), "api_response_path": attempt.get("api_response_path"), "approval_request_curl_path": attempt.get("api_request_curl_path"), "approval_response_path": attempt.get("api_response_path")})
                        rows.append(row)
                        added_step_row = True
            if not added_step_row:
                row = dict(base_row)
                row.update({"failed_step": record.get("api_stage") or "GENERAL_TERMINATION_FAILURE", "automation_error": record.get("error") or "No failed API step details captured.", "api_error": ""})
                rows.append(row)
    return rows


def save_termination_failure_report(progress, output_dir=".", basename="failure_report_termination"):
    import csv
    rows = build_termination_failure_report(progress)
    os.makedirs(output_dir, exist_ok=True)
    json_path = os.path.join(output_dir, f"{basename}.json")
    csv_path = os.path.join(output_dir, f"{basename}.csv")
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(rows, f, ensure_ascii=False, indent=2)
    try:
        with open(csv_path, "w", encoding="utf-8", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=TERMINATION_FAILURE_REPORT_FIELDS, extrasaction="ignore")
            writer.writeheader()
            writer.writerows(rows)
    except PermissionError:
        print(f"Could not update {csv_path}. Close the CSV file in Excel before reading the latest termination failure report.")


def save_termination_progress(progress):
    with open(TERMINATION_STORAGE_FILE, "w", encoding="utf-8") as f:
        json.dump(progress, f, ensure_ascii=False, indent=2)
    save_termination_failure_report(progress)

    run_output_dir = get_run_output_dir()
    if run_output_dir:
        os.makedirs(run_output_dir, exist_ok=True)
        with open(os.path.join(run_output_dir, TERMINATION_STORAGE_FILE), "w", encoding="utf-8") as f:
            json.dump(progress, f, ensure_ascii=False, indent=2)
        save_termination_failure_report(progress, run_output_dir, basename="failure_report")


def ensure_termination_progress(progress, emirates_id):
    if emirates_id not in progress:
        progress[emirates_id] = {}
    emirates_progress = progress[emirates_id]
    emirates_progress.setdefault("ejari_termination_success", {})
    emirates_progress.setdefault("ejari_termination_failure", {})
    emirates_progress.setdefault("ejari_termination_curl_only", {})
    return emirates_progress


def normalize_contract_status_name(value):
    return str(value or "").strip().lower().replace("_", " ").replace("-", " ")


def is_termination_request_status(value):
    normalized_status = normalize_contract_status_name(value)
    return normalized_status in {"termination request", "terminate request"}


def parse_selected_contract_statuses(status_input):
    if not status_input:
        return {"Active", "Termination Request"}
    if status_input.strip().lower() in {"all", "any", "*"}:
        return None
    return {item.strip() for item in status_input.split(",") if item.strip()}


def save_termination_curl(api_name, emirates_id, title, method, url, headers, payload=None, property_id=None):
    return save_api_curl_file("termination_curls", api_name, emirates_id, title, method, url, headers, payload, property_id)


def build_terminate_contract_request(contract_row_value, dld_token, bearer_token, termination_date, reason_id, admin_email):
    url = f"{DLD_PROXY_URL}/ejariservices/contract/terminate"
    headers = {
        "accept": "text/plain",
        "Content-Type": "application/json-patch+json",
        "Token": dld_token,
        "consumer-id": CONSUMER_ID,
        "Authorization": f"Bearer {bearer_token}"
    }
    payload = {
        "DataRowValue": contract_row_value,
        "TerminationDate": termination_date,
        "TerminationReasonID": reason_id,
        "TerminationOtherReason": "",
        "AdminEmailAddress": admin_email
    }
    return url, headers, payload


def terminate_contract_by_owner(contract_row_value, dld_token, bearer_token, termination_date, reason_id, admin_email):
    url, headers, payload = build_terminate_contract_request(
        contract_row_value,
        dld_token,
        bearer_token,
        termination_date,
        reason_id,
        admin_email
    )
    response = request_with_bearer("post", url, headers=headers, json=payload)
    sent_headers = final_request_auth_headers(response, headers)
    return {
        "status_code": response.status_code,
        "response": response_payload(response),
        "url": url,
        "headers": sent_headers,
        "payload": payload
    }


def build_tenant_approval_for_termination_request(contract_row_value, tenant_sequence, dld_token, bearer_token):
    url = f"{DLD_PROXY_URL}/ejariservices/contracts/{contract_row_value}/{tenant_sequence}/gettenantapprovalfortermination"
    headers = {
        "accept": "text/plain",
        "Token": dld_token,
        "consumer-id": CONSUMER_ID,
        "Authorization": f"Bearer {bearer_token}"
    }
    return url, headers


def get_tenant_approval_for_termination(contract_row_value, tenant_sequence, dld_token, bearer_token):
    url, headers = build_tenant_approval_for_termination_request(
        contract_row_value,
        tenant_sequence,
        dld_token,
        bearer_token
    )
    response = request_with_bearer("get", url, headers=headers)
    sent_headers = final_request_auth_headers(response, headers)
    return {
        "status_code": response.status_code,
        "response": response_payload(response),
        "url": url,
        "headers": sent_headers,
        "tenant_sequence": tenant_sequence
    }


def get_leased_properties_for_termination(dld_token, bearer_token):
    leased_villas = get_properties(dld_token, bearer_token, "leased", 2)
    leased_units = get_properties(dld_token, bearer_token, "leased", 3)
    for prop in leased_villas:
        prop["leased_property_type"] = "leased/2"
    for prop in leased_units:
        prop["leased_property_type"] = "leased/3"
    return leased_villas, leased_units, leased_villas + leased_units


def build_termination_contract_groups(leased_properties):
    groups = {}
    missing_contract_properties = []

    for prop in leased_properties:
        contract_row_value = prop.get("AssociatedContractRowValue")
        title = display_name(prop.get("Title")) or prop.get("PropertyNumber") or prop.get("PropertyId")
        if not contract_row_value:
            missing_contract_properties.append(prop)
            continue

        group = groups.setdefault(contract_row_value, {
            "contract_row_value": contract_row_value,
            "property_ids": [],
            "property_row_values": [],
            "titles": [],
            "contract_statuses": set(),
            "contract_end_dates": set(),
            "leased_property_types": set(),
            "properties": []
        })
        group["properties"].append(prop)
        group["property_ids"].append(prop.get("PropertyId"))
        group["property_row_values"].append(prop.get("RowValue"))
        group["titles"].append(title)
        group["contract_statuses"].add(display_name(prop.get("ContractStatus")))
        group["contract_end_dates"].add(prop.get("ContractEndDate"))
        group["leased_property_types"].add(prop.get("leased_property_type"))

    for group in groups.values():
        group["property_ids"] = [value for value in group["property_ids"] if value is not None]
        group["property_row_values"] = [value for value in group["property_row_values"] if value]
        group["titles"] = [value for value in group["titles"] if value]
        group["contract_statuses"] = sorted(value for value in group["contract_statuses"] if value)
        group["contract_end_dates"] = sorted(value for value in group["contract_end_dates"] if value)
        group["leased_property_types"] = sorted(value for value in group["leased_property_types"] if value)

    return groups, missing_contract_properties


def get_tenants_from_contract_details(contract_details_response):
    details = (contract_details_response.get("Response") or {}).get("ContractDetails") or {}
    tenants = details.get("Tenants") or []
    return tenants if isinstance(tenants, list) else []


def call_tenant_approval_for_termination_with_fallback(contract_row_value, tenant_sequence, tenant, owner_dld_token, bearer_token, approval_token_mode, emirates_id=None, title=None, property_id=None):
    attempts = []
    tenant_emirates_id = tenant.get("EmiratesId") if isinstance(tenant, dict) else None

    if approval_token_mode == "tenant_then_owner" and tenant_emirates_id:
        try:
            tenant_token_result = get_dld_token_result(str(tenant_emirates_id), bearer_token)
            if not dld_token_result_succeeded(tenant_token_result):
                raise ApiRequestError(
                    f"Tenant DLD token lookup returned no token for Emirates ID {tenant_emirates_id}: {compact_payload(tenant_token_result.get('response'))}",
                    "TENANT_DLD_TOKEN",
                    tenant_token_result.get("url"),
                    tenant_token_result.get("headers"),
                    result=tenant_token_result
                )
            tenant_dld_token = tenant_token_result["response"]["token"]
            tenant_attempt = get_tenant_approval_for_termination(
                contract_row_value,
                tenant_sequence,
                tenant_dld_token,
                bearer_token
            )
            tenant_attempt["token_source"] = "tenant"
            tenant_attempt["tenant_emirates_id"] = tenant_emirates_id
            tenant_detail_paths = save_api_result_detail_files(
                "success" if api_result_succeeded(tenant_attempt) else "failure",
                f"tenant_approval_for_termination_tenant_{tenant_sequence}_tenant_token",
                emirates_id,
                title or contract_row_value,
                "get",
                tenant_attempt,
                property_id
            )
            tenant_attempt.update(tenant_detail_paths)
            attempts.append(tenant_attempt)
            if api_result_succeeded(tenant_attempt):
                return attempts
        except Exception as exc:
            token_result = getattr(exc, "result", None)
            attempts.append({
                "token_source": "tenant",
                "tenant_emirates_id": tenant_emirates_id,
                "status_code": None,
                "error": str(exc),
                "api_stage": "TENANT_DLD_TOKEN",
                "token_result": log_safe_api_result(token_result) if token_result else None
            })

    owner_attempt = get_tenant_approval_for_termination(
        contract_row_value,
        tenant_sequence,
        owner_dld_token,
        bearer_token
    )
    owner_attempt["token_source"] = "owner"
    owner_attempt["tenant_emirates_id"] = tenant_emirates_id
    owner_detail_paths = save_api_result_detail_files(
        "success" if api_result_succeeded(owner_attempt) else "failure",
        f"tenant_approval_for_termination_tenant_{tenant_sequence}_owner_token",
        emirates_id,
        title or contract_row_value,
        "get",
        owner_attempt,
        property_id
    )
    owner_attempt.update(owner_detail_paths)
    attempts.append(owner_attempt)
    return attempts


def process_contract_termination_for_owner(emirates_id, termination_mode, progress, reason_id, admin_email, selected_statuses, approval_token_mode, max_contracts=None):
    if not confirm_process_emirates_id("contract termination", emirates_id):
        return

    print(f"\nProcessing contract termination for Emirates ID: {emirates_id}")
    emirates_progress = ensure_termination_progress(progress, emirates_id)
    success_records = emirates_progress["ejari_termination_success"]
    failure_records = emirates_progress["ejari_termination_failure"]
    curl_only_records = emirates_progress["ejari_termination_curl_only"]

    bearer_token = get_active_bearer_token(force_refresh=True)
    print(f"Generated fresh iPaaS bearer token for Emirates ID {emirates_id}")
    owner_dld_token = get_dld_token(emirates_id, bearer_token)

    leased_villas, leased_units, leased_properties = get_leased_properties_for_termination(owner_dld_token, bearer_token)
    contract_groups, missing_contract_properties = build_termination_contract_groups(leased_properties)

    emirates_progress["termination_counts"] = {
        "leased/2": len(leased_villas),
        "leased/3": len(leased_units),
        "total_leased_properties": len(leased_properties),
        "unique_contracts": len(contract_groups),
        "properties_missing_contract_row_value": len(missing_contract_properties)
    }
    save_termination_progress(progress)

    print(f"Leased properties: {len(leased_properties)} | unique contracts: {len(contract_groups)}")
    if missing_contract_properties:
        print(f"WARNING: {len(missing_contract_properties)} leased properties had no AssociatedContractRowValue")

    filtered_groups = []
    for group in contract_groups.values():
        statuses = set(group.get("contract_statuses") or [])
        is_termination_request_group = any(is_termination_request_status(status) for status in statuses)
        if selected_statuses and not is_termination_request_group and not {normalize_contract_status_name(status) for status in statuses}.intersection({normalize_contract_status_name(status) for status in selected_statuses}):
            continue
        filtered_groups.append(group)

    filtered_groups.sort(key=lambda item: (item.get("contract_end_dates") or [""], item.get("contract_row_value") or ""))
    if max_contracts is not None:
        filtered_groups = filtered_groups[:max_contracts]

    print(f"Contracts selected for termination flow: {len(filtered_groups)}")
    processed_count = 0
    success_count = 0
    failure_count = 0

    for index, group in enumerate(filtered_groups, start=1):
        contract_row_value = group["contract_row_value"]
        contract_key = contract_row_value
        title = group["titles"][0] if group["titles"] else contract_row_value
        property_id = group["property_ids"][0] if group["property_ids"] else None
        print(f"\n[{index}/{len(filtered_groups)}] Contract: {contract_row_value} | {title}")

        if success_records.get(contract_key) and termination_mode != "curl":
            processed_count += 1
            success_count += 1
            print("Already logged as termination success; skipping")
            continue

        processed_count += 1
        record = {
            "status": "STARTED",
            "emirates_id": emirates_id,
            "contract_row_value": contract_row_value,
            "property_ids": group.get("property_ids", []),
            "property_row_values": group.get("property_row_values", []),
            "titles": group.get("titles", []),
            "contract_statuses": group.get("contract_statuses", []),
            "contract_end_dates": group.get("contract_end_dates", []),
            "leased_property_types": group.get("leased_property_types", []),
            "termination_reason_id": reason_id,
            "admin_email": admin_email,
            "approval_token_mode": approval_token_mode,
            "timestamp": datetime.now().isoformat()
        }

        try:
            details_response = get_contract_details(contract_row_value, owner_dld_token, bearer_token)
            contract_details = (details_response.get("Response") or {}).get("ContractDetails") or {}
            termination_info = derive_termination_date_from_contract(contract_details)
            termination_date = termination_info["termination_date"]
            record.update(termination_info)
            record["contract_number"] = contract_details.get("ContractNumber")
            record["contract_details_status"] = display_name(contract_details.get("ContractStatus"))
            all_contract_statuses = set(record.get("contract_statuses") or [])
            if record.get("contract_details_status"):
                all_contract_statuses.add(record["contract_details_status"])
            is_already_termination_request = any(is_termination_request_status(status) for status in all_contract_statuses)
            record["owner_termination_already_submitted"] = is_already_termination_request
            tenants = get_tenants_from_contract_details(details_response)
            record["tenant_count"] = len(tenants)
            record["tenants"] = [
                {
                    "tenant_sequence": tenant_index,
                    "tenant_id": tenant.get("TenantID"),
                    "tenant_emirates_id": tenant.get("EmiratesId"),
                    "tenant_name": display_name(tenant.get("TenantName")),
                    "email": tenant.get("Email"),
                    "mobile": tenant.get("Mobile")
                }
                for tenant_index, tenant in enumerate(tenants, start=1)
                if isinstance(tenant, dict)
            ]

            terminate_url, terminate_headers, terminate_payload = build_terminate_contract_request(
                contract_row_value,
                owner_dld_token,
                bearer_token,
                termination_date,
                reason_id,
                admin_email
            )

            if termination_mode == "curl":
                termination_curl_path = save_termination_curl(
                    "terminate_contract_by_owner",
                    emirates_id,
                    title,
                    "post",
                    terminate_url,
                    terminate_headers,
                    terminate_payload,
                    property_id
                )
                approval_curl_paths = []
                approval_targets = tenants or [{}]
                for tenant_index, tenant in enumerate(approval_targets, start=1):
                    approval_url, approval_headers = build_tenant_approval_for_termination_request(
                        contract_row_value,
                        tenant_index,
                        owner_dld_token,
                        bearer_token
                    )
                    approval_curl_paths.append(save_termination_curl(
                        f"tenant_approval_for_termination_tenant_{tenant_index}",
                        emirates_id,
                        title,
                        "get",
                        approval_url,
                        approval_headers,
                        property_id=property_id
                    ))

                record["status"] = "CURL_ONLY"
                record["termination_curl_path"] = termination_curl_path
                record["approval_curl_paths"] = approval_curl_paths
                curl_only_records[contract_key] = record
                save_termination_progress(progress)
                success_count += 1
                print("Termination curls saved")
                continue

            previous_failure = failure_records.get(contract_key) or {}
            previous_terminate_result = previous_failure.get("terminate_result")
            if previous_terminate_result and api_result_succeeded(previous_terminate_result):
                terminate_result = previous_terminate_result
                print("Using previous successful terminate response from progress-termination.json")
            elif is_already_termination_request:
                terminate_result = {
                    "status_code": 200,
                    "response": {
                        "ResponseCode": "Success",
                        "ValidationErrorsList": [],
                        "Response": {
                            "AdditionalData": "Contract is already in Termination Request status; owner termination step was not called."
                        }
                    },
                    "url": None,
                    "headers": {},
                    "payload": None,
                    "skipped": True,
                    "skip_reason": "Contract is already in Termination Request status; proceeding directly to tenant approval."
                }
                print("Contract already has Termination Request status; trying tenant approval")
            else:
                terminate_result = terminate_contract_by_owner(
                    contract_row_value,
                    owner_dld_token,
                    bearer_token,
                    termination_date,
                    reason_id,
                    admin_email
                )
            if isinstance(terminate_result, dict) and terminate_result.get("url") and not terminate_result.get("skipped"):
                terminate_detail_paths = save_api_result_detail_files(
                    "success" if api_result_succeeded(terminate_result) else "failure",
                    "terminate_contract_by_owner",
                    emirates_id,
                    title,
                    "post",
                    terminate_result,
                    property_id
                )
                record["terminate_request_curl_path"] = terminate_detail_paths.get("api_request_curl_path")
                record["terminate_response_path"] = terminate_detail_paths.get("api_response_path")
            record["terminate_result"] = terminate_result

            if not api_result_succeeded(terminate_result):
                raise ApiRequestError(
                    f"Terminate Contract By Owner failed HTTP/API validation. HTTP {terminate_result.get('status_code')}: {describe_api_payload_errors(terminate_result.get('response')) or compact_payload(terminate_result.get('response'))}",
                    "TERMINATE_CONTRACT_BY_OWNER",
                    terminate_result.get("url"),
                    terminate_result.get("headers")
                )

            approval_results = []
            approval_targets = tenants or [{}]
            for tenant_index, tenant in enumerate(approval_targets, start=1):
                attempts = call_tenant_approval_for_termination_with_fallback(
                    contract_row_value,
                    tenant_index,
                    tenant,
                    owner_dld_token,
                    bearer_token,
                    approval_token_mode,
                    emirates_id=emirates_id,
                    title=title,
                    property_id=property_id
                )
                approval_results.append({
                    "tenant_sequence": tenant_index,
                    "tenant_emirates_id": tenant.get("EmiratesId") if isinstance(tenant, dict) else None,
                    "attempts": attempts
                })
            record["approval_results"] = approval_results

            failed_approval = any(not approval_result_succeeded(approval_result) for approval_result in approval_results)

            if failed_approval:
                record["status"] = "TERMINATED_APPROVAL_FAILED"
                failure_records[contract_key] = record
                failure_path = save_run_json("failure", emirates_id, title, record, property_id)
                record["failure_json_path"] = failure_path
                save_termination_progress(progress)
                failure_count += 1
                print("Terminate succeeded, but tenant approval failed")
                continue

            record["status"] = "SUCCESS"
            success_records[contract_key] = record
            failure_records.pop(contract_key, None)
            success_path = save_run_json("success", emirates_id, title, record, property_id)
            record["success_json_path"] = success_path
            save_termination_progress(progress)
            success_count += 1
            print("Termination flow succeeded")

        except Exception as exc:
            record["status"] = "ERROR"
            record["error"] = str(exc)
            record["api_stage"] = getattr(exc, "api_stage", None)
            record["timestamp"] = datetime.now().isoformat()
            failure_records[contract_key] = record
            failure_path = save_run_json("failure", emirates_id, title, record, property_id)
            record["failure_json_path"] = failure_path
            save_termination_progress(progress)
            failure_count += 1
            print(f"ERROR terminating contract {contract_row_value}: {exc}")

    print_processing_summary("contract termination", emirates_id, len(leased_properties), len(contract_groups), processed_count, success_count, failure_count)


def run_contract_termination_automation():
    termination_mode = ask_choice(
        "Contract termination mode?",
        choices=("terminate", "curl", "no"),
        default="no",
        aliases={"yes": "terminate", "run": "terminate"}
    )
    if termination_mode == "no":
        print("Skipped contract termination scripts")
        return

    if termination_mode == "terminate":
        confirmation = operator_input("Type TERMINATE to confirm actual contract termination calls", required=True).strip()
        if confirmation != "TERMINATE":
            print("Skipped contract termination scripts")
            return

    progress_mode = ask_choice("Start fresh or resume/reread existing progress-termination.json?", choices=("fresh", "resume"), default="resume")
    progress = {} if progress_mode == "fresh" else load_termination_progress()
    print(f"Progress mode selected: {progress_mode}")
    print(f"Termination progress file: {TERMINATION_STORAGE_FILE}")

    reason_id = ask_int("Termination reason ID", default=1, min_value=1)
    admin_email = operator_input("Admin email address", default="salah.h@datacellme.com").strip()

    status_input = operator_input("Contract statuses to process (comma separated, or all)", default="Active,Termination Request").strip()
    selected_statuses = parse_selected_contract_statuses(status_input)
    approval_token_mode = ask_choice("Tenant approval token mode?", choices=("owner", "tenant_then_owner"), default="owner")

    max_contracts_input = operator_input("Max unique contracts to process", default="all").strip().lower()
    if max_contracts_input in {"", "all", "none", "no"}:
        max_contracts = None
    else:
        max_contracts = int(max_contracts_input)

    start_run_output_dir("ejari_termination")
    print(f"Contract termination mode selected: {termination_mode}")
    print(f"Tenant approval token mode selected: {approval_token_mode}")
    selected_status_text = "all" if selected_statuses is None else ", ".join(sorted(selected_statuses))
    print(f"Selected statuses: {selected_status_text}")
    print(f"Run logs folder: {get_run_output_dir()}")
    ask_process_all_emirates_ids("contract termination")

    for owner_emirates_id in TERMINATION_OWNER_EMIRATES_IDS:
        process_contract_termination_for_owner(
            owner_emirates_id,
            termination_mode,
            progress,
            reason_id,
            admin_email,
            selected_statuses,
            approval_token_mode,
            max_contracts
        )


run_contract_termination_automation()


## Contract Creation And Signing

Loads vacant properties, validates each property, creates or logs the API5 curl, then verifies contract/property details and continues owner/tenant signing where required.


In [ ]:
# @title Contract creation and signing
#### ####################### CREATE CONTRACTS #################################################
##########################################################################################

def build_pending_contract_progress_record(emirates_id, contract, contract_row_value, contract_details_response=None):
    return {
        "emirates_id": emirates_id,
        "property_id": contract.get("PropertyId") if isinstance(contract, dict) else None,
        "property_row_value": contract.get("RowValue") if isinstance(contract, dict) else None,
        "title": contract_title(contract),
        "contract_row_value": contract_row_value,
        "contract_number": contract_number_from_contract(contract, contract_details_response),
        "contract_status": contract_status_name(contract),
        "source_lists": contract.get("source_lists", []) if isinstance(contract, dict) else [],
        "timestamp": datetime.now().isoformat()
    }


def continue_signing_pending_contracts_from_api(emirates_id, dld_token, bearer, progress):
    emirates_progress = ensure_emirates_progress(progress, emirates_id)
    success_records = emirates_progress["ejari_creation_success"]
    failure_records = emirates_progress["ejari_creation_failure"]

    print(f"\nChecking API contract list for previously created pending contracts for Emirates ID: {emirates_id}")
    try:
        contracts_response = get_contracts_list(dld_token, bearer)
    except Exception as exc:
        print(f"ERROR fetching contracts list for pending creation continuation: {exc}")
        print_processing_summary("contract creation pending continuation", emirates_id, 0, 0, 0, 0, 1)
        return {"total_count": 0, "unique_count": 0, "processed_count": 0, "success_count": 0, "failure_count": 1}

    response_data = contracts_response.get("Response") if isinstance(contracts_response, dict) else {}
    total_count = sum(
        1
        for source_name in ("OwnerContracts", "TenantContracts")
        for contract in (response_data.get(source_name, []) or [])
        if isinstance(contract, dict) and is_pending_contract(contract)
    )
    pending_contracts = consolidate_unique_contracts_from_contracts_response(contracts_response, only_pending=True)
    unique_count = len(pending_contracts)
    processed_count = 0
    success_count = 0
    failure_count = 0
    print(f"Pending contract list items from API: {total_count}")
    print(f"Pending unique contracts found from API: {unique_count}")

    for index, contract in enumerate(pending_contracts, start=1):
        contract_row_value = contract_row_value_from_contract(contract)
        processed_count += 1
        if not contract_row_value:
            failure_count += 1
            print("ERROR continuing pending contract: missing AssociatedContractRowValue/DataRowValue")
            continue

        property_key = str(contract.get("RowValue") or contract_row_value)
        title = contract_title(contract) or contract_row_value
        property_id = contract.get("PropertyId")
        print(f"\n[{index}/{len(pending_contracts)}] Continue pending contract: {contract.get('ContractNumber') or ''} | {contract_row_value} | {title}")

        existing_success = success_records.get(property_key)
        if existing_success and existing_success.get("status") == "SUCCESS":
            success_count += 1
            print("Skipping pending API contract already logged as SUCCESS in progress.json")
            continue

        step_results = []
        contract_details_response = None
        api_stage = "API6_CONTRACT_DETAILS"
        record = build_pending_contract_progress_record(emirates_id, contract, contract_row_value)
        record["status"] = "STARTED"

        try:
            details_result = get_contract_details_result(contract_row_value, dld_token, bearer)
            details_step = make_api_step(api_stage, details_result)
            step_results.append(details_step)
            if not api_result_succeeded(details_result):
                raise_api_step_failure(api_stage, details_result or {}, "Contract details failed before signing")

            contract_details_response = details_result.get("response")
            record.update(build_pending_contract_progress_record(emirates_id, contract, contract_row_value, contract_details_response))

            owner_signed = contract_signed_flag(contract, contract_details_response, "owner")
            tenant_signed = contract_signed_flag(contract, contract_details_response, "tenant")
            owner_participant_id = contract_owner_participant_id(contract_details_response)
            tenant_id = contract_tenant_id(contract_details_response)
            tenant_emirates_id = contract_tenant_emirates_id(contract, contract_details_response)
            record["IsContractSignedByOwner"] = owner_signed
            record["IsContractSignedByTenant"] = tenant_signed
            record["owner_participant_id"] = owner_participant_id
            record["tenant_id"] = tenant_id
            record["tenant_emirates_id"] = tenant_emirates_id

            needs_owner_sign = owner_signed is not True
            needs_tenant_sign = tenant_signed is not True
            if not needs_owner_sign and not needs_tenant_sign:
                record["status"] = "SUCCESS"
                record["continuation_status"] = "ALREADY_SIGNED"
                record["step_results"] = step_results
                success_records[property_key] = record
                failure_records.pop(property_key, None)
                save_run_json("success", emirates_id, title, record, property_id)
                save_progress(progress)
                success_count += 1
                print("Contract already appears signed by owner and tenant; logged as success")
                continue

            api_stage = "API6_DOWNLOAD_PENDING_CONTRACT"
            download_step, download_result = run_api_with_retries(
                api_stage,
                lambda: download_tenancy_contract(contract_row_value, dld_token, bearer),
                attempts=3,
                delay_seconds=2,
                success_check=download_result_has_contract_file
            )
            step_results.append(download_step)
            contract_file = extract_contract_file(download_result.get("response") if isinstance(download_result, dict) else None)
            if download_step.get("status") != "SUCCESS" or not contract_file:
                if not contract_file:
                    download_step["automation_error"] = "Download response did not include ReportData/ContractFile."
                raise_api_step_failure(api_stage, download_result or {}, "Download pending tenancy contract failed")

            if needs_owner_sign:
                if not owner_participant_id:
                    raise ApiRequestError("OwnerParticipantID is not available from contract details; cannot sign as owner.", "API7_OWNER_SIGN_CONTRACT")
                api_stage = "API7_OWNER_SIGN_CONTRACT"
                owner_sign_step, owner_sign_result = run_api_with_retries(
                    api_stage,
                    lambda: save_signed_tenancy_contract(contract_row_value, dld_token, bearer, contract_file, owner_participant_id, 1),
                    attempts=3,
                    delay_seconds=2
                )
                owner_sign_step["signed_party"] = owner_participant_id
                owner_sign_step["signed_party_type"] = 1
                save_and_attach_api_step_detail(owner_sign_step, "api7_owner_sign_contract", emirates_id, title, "post", owner_sign_result, property_id)
                step_results.append(owner_sign_step)
                if owner_sign_step.get("status") != "SUCCESS":
                    raise_api_step_failure(api_stage, owner_sign_result or {}, "Owner signing failed")
            else:
                step_results.append(make_api_step("API7_OWNER_SIGN_CONTRACT", status="SKIPPED", automation_error="Contract details says owner already signed."))

            if needs_tenant_sign:
                api_stage = "API8_TENANT_DLD_TOKEN"
                if not tenant_emirates_id:
                    raise ApiRequestError("Tenant Emirates ID is not available from contract details; cannot sign as tenant.", api_stage)
                if not tenant_id:
                    raise ApiRequestError("TenantID is not available from contract details; cannot sign as tenant.", "API9_TENANT_SIGN_CONTRACT")

                tenant_token_result = get_dld_token_result(str(tenant_emirates_id), bearer)
                if not dld_token_result_succeeded(tenant_token_result):
                    tenant_token_step = make_api_step(
                        api_stage,
                        tenant_token_result,
                        status="ERROR",
                        tenant_emirates_id=tenant_emirates_id,
                        api_error=format_failure_api_error(tenant_token_result.get("response"))
                    )
                    step_results.append(tenant_token_step)
                    raise ApiRequestError(
                        f"Tenant DLD token lookup returned no token for Emirates ID {tenant_emirates_id}: {compact_payload(tenant_token_result.get('response'))}",
                        api_stage,
                        tenant_token_result.get("url"),
                        tenant_token_result.get("headers"),
                        result=tenant_token_result
                    )
                tenant_token_step = make_api_step(api_stage, tenant_token_result, status="SUCCESS", tenant_emirates_id=tenant_emirates_id)
                step_results.append(tenant_token_step)
                tenant_dld_token = tenant_token_result["response"]["token"]

                api_stage = "API9_TENANT_SIGN_CONTRACT"
                tenant_sign_step, tenant_sign_result = run_api_with_retries(
                    api_stage,
                    lambda: save_signed_tenancy_contract(contract_row_value, tenant_dld_token, bearer, contract_file, tenant_id, 5),
                    attempts=3,
                    delay_seconds=2
                )
                tenant_sign_step["tenant_emirates_id"] = tenant_emirates_id
                tenant_sign_step["signed_party"] = tenant_id
                tenant_sign_step["signed_party_type"] = 5
                tenant_sign_step["token_source"] = "tenant"
                save_and_attach_api_step_detail(tenant_sign_step, "api9_tenant_sign_contract", emirates_id, title, "post", tenant_sign_result, property_id)
                step_results.append(tenant_sign_step)
                if tenant_sign_step.get("status") != "SUCCESS":
                    raise_api_step_failure(api_stage, tenant_sign_result or {}, "Tenant signing failed")
            else:
                step_results.append(make_api_step("API9_TENANT_SIGN_CONTRACT", status="SKIPPED", automation_error="Contract details says tenant already signed."))

            record["status"] = "SUCCESS"
            record["continuation_status"] = "SIGNED_FROM_API_PENDING_LIST"
            record["step_results"] = step_results
            success_records[property_key] = record
            failure_records.pop(property_key, None)
            save_run_json("success", emirates_id, title, record, property_id)
            save_progress(progress)
            success_count += 1
            print("Pending contract signing continuation succeeded")

        except Exception as exc:
            record["status"] = "ERROR"
            record["api_stage"] = getattr(exc, "api_stage", None) or api_stage
            record["error"] = str(exc)
            record["step_results"] = step_results
            record["timestamp"] = datetime.now().isoformat()
            failure_records[property_key] = record
            failure_path = save_run_json("failure", emirates_id, title, record, property_id)
            record["failure_json_path"] = failure_path
            save_progress(progress)
            failure_count += 1
            print(f"ERROR continuing pending contract {contract_row_value}: {exc}")

    print_processing_summary("contract creation pending continuation", emirates_id, total_count, unique_count, processed_count, success_count, failure_count)
    return {
        "total_count": total_count,
        "unique_count": unique_count,
        "processed_count": processed_count,
        "success_count": success_count,
        "failure_count": failure_count
    }


# ================= MAIN FLOW =================
def process(emirates_id, creation_mode="create", progress=None, continue_pending_from_api=False):
    if not confirm_process_emirates_id("contract creation", emirates_id):
        return

    if progress is None:
        progress = load_progress()
    run_output_dir = get_run_output_dir()
    creation_mode = (creation_mode or "create").strip().lower()
    log_create_curl_only = creation_mode == "curl"

    emirates_progress = ensure_emirates_progress(progress, emirates_id)
    success_records = emirates_progress["ejari_creation_success"]
    failure_records = emirates_progress["ejari_creation_failure"]
    curl_only_records = emirates_progress["ejari_creation_curl_only"]

    print(f"\nProcessing Emirates ID: {emirates_id}")
    print(f"Run logs folder: {run_output_dir}")
    if log_create_curl_only:
        print("Creation mode: API5 create curl only. API5 create will not be called.")

    tenant_lookup = select_tenant_lookup()
    print(f"Tenant selected for owner {emirates_id}: {tenant_lookup['name']} | EID: {tenant_lookup['emirates_id']} | DOB: {tenant_lookup['dob']}")

    # ALWAYS rerun API1-3
    bearer = get_active_bearer_token(force_refresh=True)
    print(f"Generated fresh iPaaS bearer token for Emirates ID {emirates_id}")
    dld_token = get_dld_token(emirates_id, bearer)
    if continue_pending_from_api and not log_create_curl_only:
        continue_signing_pending_contracts_from_api(emirates_id, dld_token, bearer, progress)
    tenant = get_tenant_person(tenant_lookup["emirates_id"], tenant_lookup["dob"], dld_token, bearer)
    tenant_name_value = tenant.get("TenantName", {})
    tenant_name = tenant_name_value.get("EnglishName") if isinstance(tenant_name_value, dict) else str(tenant_name_value or "")
    selected_tenant_emirates_id = str(tenant.get("EmiratesId") or tenant_lookup["emirates_id"])
    selected_tenant_id = tenant.get("TenantID") or tenant.get("TenantId") or tenant.get("tenantId")
    print(f"Loaded tenant details for tenant Emirates ID {selected_tenant_emirates_id}: {tenant_name}")

    all_properties = []
    vacantVillas = annotate_owner_asset_source(get_properties(dld_token, bearer, "vacant", 2), "vacant", 2)
    vacantUnits = annotate_owner_asset_source(get_properties(dld_token, bearer, "vacant", 3), "vacant", 3)
    # Vacant properties
    all_properties.extend(vacantVillas)
    all_properties.extend(vacantUnits)

    backfill_progress_property_ids(emirates_progress, all_properties)

    emirates_progress["property_counts"] = {
        "vacant/2": len(vacantVillas),
        "vacant/3": len(vacantUnits),
        "total_vacant": len(vacantVillas) + len(vacantUnits)
    }
    save_progress(progress)

    print(f"Properties loaded for Emirates ID {emirates_id}: vacant/2={len(vacantVillas)}, vacant/3={len(vacantUnits)}, total_vacant={len(vacantVillas) + len(vacantUnits)}")

    for prop in all_properties:
        title = prop["Title"]["EnglishName"]
        property_id = prop.get("PropertyId")
        row_value = prop.get("RowValue") or property_id
        property_key = str(row_value)
        property_list_type = prop.get("_owner_asset_list_type") or "vacant"
        property_type_id = prop.get("_owner_asset_property_type_id") or prop.get("PropertyTypeId") or prop.get("PropertyTypeID")

        print(f"\nProperty: {title} | \nProperty ID: {property_id} | \nRow Value: {row_value}")

        prop_data = success_records.get(property_key)
        print(f"\nProperty Data found against row value")
        # Skip only if SUCCESS
        if prop_data and prop_data.get("status") == "SUCCESS" and not log_create_curl_only:
            prop_data["title"] = title
            prop_data["status"] = "SUCCESS"
            prop_data["property_id"] = property_id
            prop_data["property_row_value"] = row_value
            success_records[property_key] = prop_data
            save_run_json("success", emirates_id, title, prop_data, property_id)
            print("Skipping (already SUCCESS)")
            save_progress(progress)
            continue

        step_results = []
        try:
            ejari_id = None
            contract_row_value = None
            contract_number = None
            tenant_emirates_id = selected_tenant_emirates_id
            tenant_id = selected_tenant_id
            owner_participant_id = None
            api5_request_curl_path = None
            api5_response_path = None
            contract_details_request_curl_path = None
            contract_details_response_path = None
            property_detail_request_curl_path = None
            property_detail_response_path = None
            property_detail_list_type_used = None
            owner_signed = None
            tenant_signed = None
            fail_api4_id = None
            fail_id = None
            api_stage = "API4_VALIDATE_PROPERTY"
            ejari_id = validate_property(row_value, dld_token, bearer)
            print("API4 Ejari ID:", ejari_id)

            api_stage = "API5_CREATE_CONTRACT"
            if log_create_curl_only:
                url, headers, payload = build_create_contract_request(ejari_id, dld_token, bearer, tenant)
                curl_path = save_create_api5_curl(emirates_id, title, url, headers, payload, property_id)
                curl_only_records[property_key] = {
                    "ejari_property_id": ejari_id,
                    "property_id": property_id,
                    "property_row_value": row_value,
                    "property_list_type": property_list_type,
                    "property_type_id": property_type_id,
                    "title": title,
                    "status": "CURL_ONLY",
                    "api_stage": "API5_CREATE_CONTRACT_CURL_ONLY",
                    "create_curl_path": curl_path,
                    "timestamp": str(datetime.now())
                }
                print("API5 create curl saved:", curl_path)
                save_progress(progress)
                continue

            api5_result = create_contract(ejari_id, dld_token, bearer, tenant)
            step_results.append(make_api_step(
                "API5_CREATE_CONTRACT",
                api5_result,
                tenant_emirates_id=tenant_emirates_id,
                tenant_id=tenant_id
            ))
            print(f"API5 create completed with HTTP {api5_result.get('status_code')}. Full request/response will be saved to run detail files.")

            if api5_result.get("status_code", 200) >= 400:
                detail_paths = save_api5_detail_files(
                    "failure",
                    emirates_id,
                    title,
                    api5_result["url"],
                    api5_result["headers"],
                    api5_result["payload"],
                    api5_result,
                    property_id
                )
                fail_id = detail_paths["api5_request_curl_path"]
                api5_request_curl_path = detail_paths["api5_request_curl_path"]
                api5_response_path = detail_paths["api5_response_path"]
                attach_api5_detail_paths_to_latest_step(step_results, api5_result, detail_paths)
                raise ApiRequestError(f"API5 HTTP {api5_result['status_code']}: {compact_payload(api5_result['response'])} | fail_id={fail_id}", "API5_CREATE_CONTRACT", api5_result.get("url"), api5_result.get("headers"), result=api5_result)

            api5_response = api5_result["response"]
            if not isinstance(api5_response, dict):
                detail_paths = save_api5_detail_files(
                    "failure",
                    emirates_id,
                    title,
                    api5_result["url"],
                    api5_result["headers"],
                    api5_result["payload"],
                    api5_result,
                    property_id
                )
                fail_id = detail_paths["api5_request_curl_path"]
                api5_request_curl_path = detail_paths["api5_request_curl_path"]
                api5_response_path = detail_paths["api5_response_path"]
                attach_api5_detail_paths_to_latest_step(step_results, api5_result, detail_paths)
                raise ApiRequestError(f"API5 returned non-JSON response: {compact_payload(api5_response)} | fail_id={fail_id}", "API5_CREATE_CONTRACT", api5_result.get("url"), api5_result.get("headers"), result=api5_result)

            api5_errors = api_payload_errors(api5_response)
            if api5_errors:
                detail_paths = save_api5_detail_files(
                    "failure",
                    emirates_id,
                    title,
                    api5_result["url"],
                    api5_result["headers"],
                    api5_result["payload"],
                    api5_result,
                    property_id
                )
                fail_id = detail_paths["api5_request_curl_path"]
                api5_request_curl_path = detail_paths["api5_request_curl_path"]
                api5_response_path = detail_paths["api5_response_path"]
                attach_api5_detail_paths_to_latest_step(step_results, api5_result, detail_paths)
                first_error = api5_errors[0]
                raise ApiRequestError(f"{get_api5_error_message(first_error)} | fail_id={fail_id}", "API5_CREATE_CONTRACT", api5_result.get("url"), api5_result.get("headers"), result=api5_result)

            detail_paths = save_api5_detail_files(
                "success",
                emirates_id,
                title,
                api5_result["url"],
                api5_result["headers"],
                api5_result["payload"],
                api5_result,
                property_id
            )
            api5_request_curl_path = detail_paths["api5_request_curl_path"]
            api5_response_path = detail_paths["api5_response_path"]
            attach_api5_detail_paths_to_latest_step(step_results, api5_result, detail_paths)
            print(f"API5 create detail files saved: {api5_request_curl_path}; {api5_response_path}")

            contract_row_value = get_created_contract_row_value(api5_response)
            contract_number = get_created_contract_number(api5_response)

            api_stage = "API6_CONTRACT_DETAILS_AFTER_SUBMISSION"
            details_step, details_result = run_api_with_retries(
                api_stage,
                lambda: get_contract_details_result(contract_row_value, dld_token, bearer),
                attempts=5,
                delay_seconds=2,
                success_check=contract_details_result_succeeded
            )
            details_detail_paths = save_api_result_detail_files(
                "success" if details_step.get("status") == "SUCCESS" else "failure",
                "api6_contract_details_after_submission",
                emirates_id,
                title,
                "get",
                details_result,
                property_id
            )
            contract_details_request_curl_path = details_detail_paths.get("api_request_curl_path")
            contract_details_response_path = details_detail_paths.get("api_response_path")
            attach_api_detail_paths_to_step(details_step, details_result, details_detail_paths)
            if details_step.get("status") != "SUCCESS":
                details_step["automation_error"] = "Unable to load contract detail after contract submission."
            step_results.append(details_step)
            if details_step.get("status") != "SUCCESS":
                raise_api_step_failure(api_stage, details_result or {}, "Unable to load contract detail after contract submission")

            api_stage = "API_OWNER_ASSET_PROPERTY_DETAIL_AFTER_SUBMISSION"
            property_detail_step, property_detail_result = run_api_with_retries(
                api_stage,
                lambda: get_owner_asset_property_detail_result(property_list_type, property_type_id, row_value, dld_token, bearer),
                attempts=5,
                delay_seconds=2,
                success_check=property_detail_result_succeeded
            )
            property_detail_paths = save_api_result_detail_files(
                "success" if property_detail_step.get("status") == "SUCCESS" else "failure",
                "api_owner_asset_property_detail_after_submission",
                emirates_id,
                title,
                "get",
                property_detail_result,
                property_id
            )
            property_detail_request_curl_path = property_detail_paths.get("api_request_curl_path")
            property_detail_response_path = property_detail_paths.get("api_response_path")
            property_detail_list_type_used = property_detail_result.get("owner_asset_list_type") if isinstance(property_detail_result, dict) else None
            attach_api_detail_paths_to_step(property_detail_step, property_detail_result, property_detail_paths)
            if property_detail_step.get("status") != "SUCCESS":
                property_detail_step["automation_error"] = "Unable to load property detail after contract submission."
            step_results.append(property_detail_step)
            if property_detail_step.get("status") != "SUCCESS":
                raise_api_step_failure(api_stage, property_detail_result or {}, "Unable to load property detail after contract submission")

            contract_details_response = details_result.get("response")
            contract_number = contract_number or contract_number_from_contract({}, contract_details_response)
            owner_signed = contract_signed_flag({}, contract_details_response, "owner")
            tenant_signed = contract_signed_flag({}, contract_details_response, "tenant")
            owner_participant_id = contract_owner_participant_id(contract_details_response)
            tenant_id = contract_tenant_id(contract_details_response)
            tenant_emirates_id = contract_tenant_emirates_id({}, contract_details_response) or tenant.get("EmiratesId")

            api_stage = "API6_DOWNLOAD_PENDING_CONTRACT"
            download_step, download_result = run_api_with_retries(
                api_stage,
                lambda: download_tenancy_contract(contract_row_value, dld_token, bearer),
                attempts=3,
                delay_seconds=2,
                success_check=download_result_has_contract_file
            )
            step_results.append(download_step)
            contract_file = extract_contract_file(download_result.get("response") if isinstance(download_result, dict) else None)
            if download_step.get("status") != "SUCCESS" or not contract_file:
                if not contract_file:
                    download_step["automation_error"] = "Download response did not include ReportData/ContractFile."
                raise_api_step_failure(api_stage, download_result or {}, "Download pending tenancy contract failed")

            if owner_signed is not True:
                if not owner_participant_id:
                    raise ApiRequestError("OwnerParticipantID is not available from contract details; cannot sign as owner.", "API7_OWNER_SIGN_CONTRACT")
                api_stage = "API7_OWNER_SIGN_CONTRACT"
                owner_sign_step, owner_sign_result = run_api_with_retries(
                    api_stage,
                    lambda: save_signed_tenancy_contract(contract_row_value, dld_token, bearer, contract_file, owner_participant_id, 1),
                    attempts=3,
                    delay_seconds=2
                )
                owner_sign_step["signed_party"] = owner_participant_id
                owner_sign_step["signed_party_type"] = 1
                save_and_attach_api_step_detail(owner_sign_step, "api7_owner_sign_contract", emirates_id, title, "post", owner_sign_result, property_id)
                step_results.append(owner_sign_step)
                if owner_sign_step.get("status") != "SUCCESS":
                    raise_api_step_failure(api_stage, owner_sign_result or {}, "Owner signing failed")
            else:
                step_results.append(make_api_step("API7_OWNER_SIGN_CONTRACT", status="SKIPPED", automation_error="Contract details says owner already signed."))

            if tenant_signed is not True:
                api_stage = "API8_TENANT_DLD_TOKEN"
                if not tenant_emirates_id:
                    raise ApiRequestError("Tenant Emirates ID is not available from contract details; cannot sign as tenant.", api_stage)
                if not tenant_id:
                    raise ApiRequestError("TenantID is not available from contract details; cannot sign as tenant.", "API9_TENANT_SIGN_CONTRACT")

                tenant_token_result = get_dld_token_result(str(tenant_emirates_id), bearer)
                if not dld_token_result_succeeded(tenant_token_result):
                    tenant_token_step = make_api_step(
                        api_stage,
                        tenant_token_result,
                        status="ERROR",
                        tenant_emirates_id=tenant_emirates_id,
                        api_error=format_failure_api_error(tenant_token_result.get("response"))
                    )
                    step_results.append(tenant_token_step)
                    raise ApiRequestError(
                        f"Tenant DLD token lookup returned no token for Emirates ID {tenant_emirates_id}: {compact_payload(tenant_token_result.get('response'))}",
                        api_stage,
                        tenant_token_result.get("url"),
                        tenant_token_result.get("headers"),
                        result=tenant_token_result
                    )
                tenant_token_step = make_api_step(api_stage, tenant_token_result, status="SUCCESS", tenant_emirates_id=tenant_emirates_id)
                step_results.append(tenant_token_step)
                tenant_dld_token = tenant_token_result["response"]["token"]

                api_stage = "API9_TENANT_SIGN_CONTRACT"
                tenant_sign_step, tenant_sign_result = run_api_with_retries(
                    api_stage,
                    lambda: save_signed_tenancy_contract(contract_row_value, tenant_dld_token, bearer, contract_file, tenant_id, 5),
                    attempts=3,
                    delay_seconds=2
                )
                tenant_sign_step["tenant_emirates_id"] = tenant_emirates_id
                tenant_sign_step["signed_party"] = tenant_id
                tenant_sign_step["signed_party_type"] = 5
                tenant_sign_step["token_source"] = "tenant"
                save_and_attach_api_step_detail(tenant_sign_step, "api9_tenant_sign_contract", emirates_id, title, "post", tenant_sign_result, property_id)
                step_results.append(tenant_sign_step)
                if tenant_sign_step.get("status") != "SUCCESS":
                    raise_api_step_failure(api_stage, tenant_sign_result or {}, "Tenant signing failed")
            else:
                step_results.append(make_api_step("API9_TENANT_SIGN_CONTRACT", status="SKIPPED", automation_error="Contract details says tenant already signed."))

            success_records[property_key] = {
                "ejari_property_id": ejari_id,
                "property_id": property_id,
                "property_row_value": row_value,
                "property_list_type": property_list_type,
                "property_type_id": property_type_id,
                "property_detail_list_type_used": property_detail_list_type_used,
                "title": title,
                "status": "SUCCESS",
                "contract_row_value": contract_row_value,
                "contract_number": contract_number,
                "tenant_emirates_id": tenant_emirates_id,
                "tenant_id": tenant_id,
                "tenant_name": tenant_name,
                "owner_participant_id": owner_participant_id,
                "IsContractSignedByOwner": owner_signed,
                "IsContractSignedByTenant": tenant_signed,
                "api5_request_curl_path": api5_request_curl_path,
                "api5_response_path": api5_response_path,
                "contract_details_request_curl_path": contract_details_request_curl_path,
                "contract_details_response_path": contract_details_response_path,
                "property_detail_request_curl_path": property_detail_request_curl_path,
                "property_detail_response_path": property_detail_response_path,
                "step_results": step_results,
                "timestamp": str(datetime.now())
            }
            save_run_json("success", emirates_id, title, success_records[property_key], property_id)
            failure_records.pop(property_key, None)

        except Exception as e:
            if api_stage == "API4_VALIDATE_PROPERTY" and getattr(e, "url", None) and getattr(e, "headers", None):
                fail_api4_id = save_failed_api4(emirates_id, title, e.url, e.headers, property_id)

            print("ERROR:", str(e))
            failure_records[property_key] = {
                "ejari_property_id": ejari_id,
                "property_id": property_id,
                "property_row_value": row_value,
                "property_list_type": property_list_type,
                "property_type_id": property_type_id,
                "property_detail_list_type_used": property_detail_list_type_used,
                "title": title,
                "contract_row_value": contract_row_value,
                "contract_number": contract_number,
                "tenant_emirates_id": tenant_emirates_id,
                "tenant_id": tenant_id,
                "tenant_name": tenant_name,
                "owner_participant_id": owner_participant_id,
                "IsContractSignedByOwner": owner_signed,
                "IsContractSignedByTenant": tenant_signed,
                "status": "ERROR",
                "api_stage": getattr(e, "api_stage", None) or api_stage,
                "error": str(e),
                "failed_api4_id": fail_api4_id,
                "failed_api5_id": fail_id,
                "api5_request_curl_path": api5_request_curl_path,
                "api5_response_path": api5_response_path,
                "contract_details_request_curl_path": contract_details_request_curl_path,
                "contract_details_response_path": contract_details_response_path,
                "property_detail_request_curl_path": property_detail_request_curl_path,
                "property_detail_response_path": property_detail_response_path,
                "step_results": step_results,
                "failure_json_path": None,
                "timestamp": str(datetime.now())
            }
            failure_path = save_run_json("failure", emirates_id, title, failure_records[property_key], property_id)
            failure_records[property_key]["failure_json_path"] = failure_path

        save_progress(progress)

    unique_property_keys = {str(prop.get("RowValue") or prop.get("PropertyId")) for prop in all_properties}
    processed_count = len(success_records) + len(failure_records) + len(curl_only_records)
    print_processing_summary("contract creation", emirates_id, len(all_properties), len(unique_property_keys), processed_count, len(success_records), len(failure_records))
    print("\nDONE")

TENANT_LOOKUP_OPTIONS = {
    "1": {
        "name": "Natasha Test",
        "emirates_id": "784197265140323",
        "dob": "1963-01-01"
    },
    "2": {
        "name": "Inaya Test",
        "emirates_id": "784199515347708",
        "dob": "1995-01-11"
    },
    "3": {
        "name": "Dubai Now Test",
        "emirates_id": "784195279540512",
        "dob": "1976-10-27"
    },
    "4": {
        "name": "Slama Hammed",
        "emirates_id": "784199668638416",
        "dob": "1996-05-17"
    },
    "5": {
        "name": "Aqeel",
        "emirates_id": "784198721116758",
        "dob": "1987-09-18"
    }
}

def normalize_tenant_dob(value):
    value = str(value or "").strip()
    if not value:
        return ""

    cleaned_value = value.replace("Sept", "Sep").replace("sept", "sep")
    for date_format in ("%Y-%m-%d", "%d/%b/%Y", "%d/%B/%Y", "%d %b %Y", "%d %B %Y", "%d/%m/%Y", "%d-%m-%Y"):
        try:
            return datetime.strptime(cleaned_value, date_format).strftime("%Y-%m-%d")
        except ValueError:
            continue

    raise ValueError("Tenant DOB must be in YYYY-MM-DD format, or a date like 11/Jan/1995.")

def select_tenant_lookup():
    print("\nTenant options:")
    for option_number, tenant_option in TENANT_LOOKUP_OPTIONS.items():
        dob_text = tenant_option.get("dob") or "DOB required"
        print(f"{option_number}. {tenant_option['name']} | EID: {tenant_option['emirates_id']} | DOB: {dob_text}")
    print("6. Other")

    selected_option = ask_choice("Select tenant option", choices=("1", "2", "3", "4", "5", "6"), default="5")
    if selected_option == "6":
        tenant_name = "Other"
        tenant_emirates_id = operator_input("Tenant Emirates ID for create payload", required=True).strip()
        tenant_dob = operator_input("Tenant DOB for create payload (YYYY-MM-DD)", required=True).strip()
    else:
        tenant_option = TENANT_LOOKUP_OPTIONS.get(selected_option)
        if tenant_option is None:
            raise ValueError("Invalid tenant option. Choose 1, 2, 3, 4, 5, or 6.")
        tenant_name = tenant_option["name"]
        tenant_emirates_id = tenant_option["emirates_id"]
        tenant_dob = tenant_option.get("dob")
        if not tenant_dob:
            tenant_dob = operator_input(f"Tenant DOB for {tenant_name} (YYYY-MM-DD)", required=True).strip()

    tenant_dob = normalize_tenant_dob(tenant_dob)
    if not tenant_emirates_id or not tenant_dob:
        raise ValueError("Tenant Emirates ID and DOB are required to build create payloads.")

    return {
        "name": tenant_name,
        "emirates_id": str(tenant_emirates_id).strip(),
        "dob": tenant_dob
    }

# ================= RUN =================
creation_mode = ask_choice(
    "Contract creation mode?",
    choices=("create", "curl", "no"),
    default="no",
    aliases={"yes": "create"}
)
if creation_mode in {"create", "curl"}:
    print(f"Contract creation mode selected: {creation_mode}")

    progress_mode = ask_choice("Start fresh or resume/reread existing progress.json?", choices=("fresh", "resume"), default="resume")
    progress = {} if progress_mode == "fresh" else load_progress()
    print(f"Progress mode selected: {progress_mode}")

    continue_pending_from_api = False
    if creation_mode == "create":
        continue_pending_from_api = ask_yes_no("Continue signing previously created pending contracts from API?", default=False)
    print(f"Continue pending contracts from API: {continue_pending_from_api}")

    start_run_output_dir()
    ask_process_all_emirates_ids("contract creation")
    for owner_emirates_id in OWNER_EMIRATES_IDS:
        process(owner_emirates_id, creation_mode, progress, continue_pending_from_api)
else:
    print("Skipped contract creation scripts")


## DEWA Premise Diagnostic

Optional diagnostic section for contract/DEWA premise status checks. Use the standalone DEWA audit notebook for larger premise reports.


In [ ]:
# @title DEWA premise diagnostic
###################################################################################
# DEWA premise statuses
def getDewaPremiseStatuses(emirates_id, selected_statuses=None):
    if not confirm_process_emirates_id("DEWA premise status", emirates_id):
        return

    print(f"\nProcessing Emirates ID: {emirates_id}")

    bearer_token = get_active_bearer_token(force_refresh=True)
    print(f"Generated fresh iPaaS bearer token for Emirates ID {emirates_id}")
    dld_token = get_dld_token(emirates_id, bearer_token)

    contracts_response = get_contracts_list(dld_token, bearer_token)

    owner = contracts_response.get("Response", {}).get("OwnerContracts", [])
    tenant = contracts_response.get("Response", {}).get("TenantContracts", [])

    all_contracts = owner + tenant

    if not all_contracts:
        print("No contracts found")
        return

    # ---- Collect statuses ----
    status_map = {}
    for c in all_contracts:
        status = c["ContractStatus"]["EnglishName"]
        status_map.setdefault(status, []).append(c)

    print("\nAvailable statuses:")
    for s in status_map.keys():
        print(f"- {s}")

    selected_status_text = "all" if selected_statuses is None else ", ".join(sorted(selected_statuses))
    print(f"\nSelected statuses: {selected_status_text}")

    filtered_contracts = [
        c for c in all_contracts
        if selected_statuses is None or c["ContractStatus"]["EnglishName"] in selected_statuses
    ]

    print(f"\nFiltered contracts: {len(filtered_contracts)}")

    if not filtered_contracts:
        return

    # ---- DEWA token once ----
    dewa_token = get_dewa_token(bearer_token)

    # ---- Process each contract ----
    processed_pairs = set()
    for c in filtered_contracts:
        try:
            title = c["Title"]["EnglishName"]
            contract_row_value = c["AssociatedContractRowValue"]

            details = get_contract_details(contract_row_value, dld_token, bearer_token)

            contract_details = details["Response"]["ContractDetails"]

            contract_number = contract_details["ContractNumber"]
            start_date = contract_details["ContractStartDate"]
            end_date = contract_details["ContractEndDate"]

            premise_no = contract_details["AssociatedProperties"][0]["DewaAccount"][0]["AccountNumber"]

            key = (contract_number, premise_no)
            if key in processed_pairs:
                print(f"Skipping duplicate: {key}")
                continue

            processed_pairs.add(key)

            premise_resp = premise_status_check(contract_number, premise_no, bearer_token)

            contract_type = "Owner" if c in owner else "Tenant"
            contract_status = c["ContractStatus"]["EnglishName"]

            item = premise_resp["MT_PremiseStatusCheck_Resp"]["Item"][0]
            contract_act_number = item.get("ContractActNo")

            dewa_resp = None
            if contract_act_number:
                dewa_resp = dewa_enquiry(contract_act_number, dewa_token, bearer_token)

            tenant = contract_details.get("Tenants", [{}])[0]
            tenant_name = tenant.get("TenantName", {}).get("EnglishName")
            tenant_emirates_id = tenant.get("EmiratesId")
            tenant_id = tenant.get("TenantID")

            print("\n-----------------------------------")
            print(f"Title: {title}")
            print(f"Contract Number: {contract_number}")
            print(f"Type: {contract_type}")
            print(f"Status: {contract_status}")
            print(f"Row Value: {contract_row_value}")
            print(f"Start: {start_date}")
            print(f"End: {end_date}")
            print(f"Tenant Name: {tenant_name}")
            print(f"Tenant Emirates ID: {tenant_emirates_id}")
            print(f"Tenant ID: {tenant_id}")
            print(f"Premise Status: {item}")
            print(f"DEWA Enquiry: {dewa_resp}")

        except Exception as e:
            print(f"\nERROR processing contract: {e}")

run_check = ask_yes_no("Do you want to run DEWA premises diagnostic?", default=False)
if run_check:
    status_input = operator_input("Contract statuses to process (comma separated, or all)", default="Active,Termination Request").strip()
    if status_input.lower() in {"all", "any", "*"}:
        selected_statuses = None
    elif status_input:
        selected_statuses = {status.strip() for status in status_input.split(",") if status.strip()}
    else:
        selected_statuses = {"Active", "Termination Request"}

    ask_process_all_emirates_ids("DEWA premise status")
    for owner_emirates_id in OWNER_EMIRATES_IDS:
        getDewaPremiseStatuses(owner_emirates_id, selected_statuses)
else:
    print("Skipped DEWA premises diagnostic.")